# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION WITH PCA DIMENSIONALITY REDUCTION
# Creates two ChromaDB collections: full embeddings (768D) and PCA reduced (50D)
# Using MEAN POOLING - chroma_db_mean_content_full & chroma_db_mean_content_PCA50
# =============================================================================

In [1]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION WITH PCA DIMENSIONALITY REDUCTION
# Creates two ChromaDB collections: full embeddings (768D) and PCA reduced (64D)
# Using MEAN POOLING - chroma_db_mean_content_full & chroma_db_mean_content_PCA64
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time
from sklearn.decomposition import PCA
import pickle

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in two ChromaDB collections:
    1. Full embeddings (768 dimensions)
    2. PCA-reduced embeddings (64 dimensions)

    Designed for large datasets (10,000+ records) with semantic search on content.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "./chroma_db_mean_content_full_claude",
                 chroma_db_path_pca: str = "./chroma_db_mean_content_PCA64_claude",
                 pca_dimensions: int = 64):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path_full: Path to store ChromaDB with full embeddings
            chroma_db_path_pca: Path to store ChromaDB with PCA-reduced embeddings
            pca_dimensions: Number of dimensions for PCA reduction (default: 64)
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_pca = Path(chroma_db_path_pca)
        self.pca_dimensions = pca_dimensions
        self.pca_model = None

        # Create directories
        self.chroma_db_path_full.mkdir(exist_ok=True)
        self.chroma_db_path_pca.mkdir(exist_ok=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Initialize ChromaDB clients for both collections
        print(f"Initializing ChromaDB for full embeddings at: {self.chroma_db_path_full}")
        self.client_full = chromadb.PersistentClient(
            path=str(self.chroma_db_path_full)
        )

        print(f"Initializing ChromaDB for PCA embeddings at: {self.chroma_db_path_pca}")
        self.client_pca = chromadb.PersistentClient(
            path=str(self.chroma_db_path_pca)
        )

        # Create or get collections for storing embeddings
        self.collection_full = self.client_full.get_or_create_collection(
            name="urdu_news_embeddings_full_mean_claude",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )

        self.collection_pca = self.client_pca.get_or_create_collection(
            name="urdu_news_embeddings_pca64_mean_claude",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )

    def mean_pooling(self, model_output, attention_mask):
        """
        Apply mean pooling to model output to get sentence embeddings.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            Mean pooled embeddings (batch_size, embedding_dim)
        """
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512,
                                    chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for a single text using mean pooling.

        Args:
            text: Input Urdu text (content column)
            max_length: Maximum tokens per chunk (default: 512)
            chunk_overlap: Overlap between chunks to maintain context (default: 50)

        Returns:
            Embedding vector as numpy array (768,)
        """
        tokens = self.tokenizer.encode(text, add_special_tokens=True)

        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long articles: Split into overlapping chunks
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def fit_pca(self, embeddings_array: np.ndarray) -> None:
        """
        Fit PCA model on the full embeddings.

        Args:
            embeddings_array: Array of shape (n_samples, embedding_dim)
        """
        print(f"\n{'='*70}")
        print(f"FITTING PCA MODEL ({self.pca_dimensions} dimensions)")
        print(f"{'='*70}")
        print(f"Input shape: {embeddings_array.shape}")

        self.pca_model = PCA(n_components=self.pca_dimensions, random_state=42)
        self.pca_model.fit(embeddings_array)

        explained_variance = np.sum(self.pca_model.explained_variance_ratio_) * 100
        print(f"✓ PCA fitted successfully!")
        print(f"Explained variance: {explained_variance:.2f}%")
        print(f"Original dimensions: {embeddings_array.shape[1]}")
        print(f"Reduced dimensions: {self.pca_dimensions}")

        # Save PCA model
        pca_model_path = self.chroma_db_path_pca / "pca_model.pkl"
        with open(pca_model_path, 'wb') as f:
            pickle.dump(self.pca_model, f)
        print(f"✓ PCA model saved to: {pca_model_path}")

    def apply_pca(self, embeddings_array: np.ndarray) -> np.ndarray:
        """
        Apply PCA transformation to embeddings.

        Args:
            embeddings_array: Array of shape (n_samples, embedding_dim)

        Returns:
            Reduced embeddings of shape (n_samples, pca_dimensions)
        """
        if self.pca_model is None:
            raise ValueError("PCA model not fitted. Call fit_pca first.")

        return self.pca_model.transform(embeddings_array)

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       content_column: str = "content",
                                       headline_column: str = "Headline",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles and store in two ChromaDB collections:
        1. Full embeddings (768D)
        2. PCA-reduced embeddings (64D)

        Args:
            df: Dataframe containing articles
            content_column: Column containing article content
            headline_column: Column containing article headline
            category_column: Column containing article category
        """
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Content column: '{content_column}' (used for semantic search)")
        print(f"Headline column: '{headline_column}' (stored as metadata)")
        print(f"Category column: '{category_column}' (stored as metadata)")
        print(f"Pooling method: MEAN POOLING")
        print(f"Full embeddings DB: chroma_db_mean_content_full")
        print(f"PCA embeddings DB: chroma_db_mean_content_PCA64")
        print(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data
        ids = []
        embeddings_full = []
        metadatas = []
        documents = []

        # Step 1: Generate full embeddings
        print("STEP 1: Generating full embeddings (768D)...")
        print("="*70)

        for idx, row in df.iterrows():
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed
                eta = (total_articles - idx - 1) / articles_per_sec
                print(f"Processed {idx + 1}/{total_articles} articles "
                      f"({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)")

            content_text = str(row[content_column])

            if len(content_text.strip()) == 0:
                print(f"Warning: Skipping article {idx} - empty content")
                continue

            try:
                # Generate full embedding
                embedding = self.generate_embedding_for_text(content_text)

                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings_full.append(embedding)
                documents.append(content_text[:500])

                metadata = {
                    "article_index": idx,
                    "headline": str(row.get(headline_column, "Unknown")),
                    "category": str(row.get(category_column, "Unknown")),
                    "content_length": len(content_text),
                    "pooling_method": "mean_pooling"
                }
                metadatas.append(metadata)

            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        # Convert to numpy array for PCA
        embeddings_full_array = np.array(embeddings_full)

        # Step 2: Fit and apply PCA
        self.fit_pca(embeddings_full_array)

        print(f"\n{'='*70}")
        print("STEP 2: Applying PCA transformation...")
        print("="*70)

        embeddings_pca = self.apply_pca(embeddings_full_array)
        print(f"✓ PCA transformation complete!")
        print(f"Reduced embeddings shape: {embeddings_pca.shape}")

        # Step 3: Store full embeddings in ChromaDB
        print(f"\n{'='*70}")
        print(f"STEP 3: STORING FULL EMBEDDINGS (768D) IN CHROMADB...")
        print(f"{'='*70}")

        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            # Add metadata indicating this is full embedding
            batch_metadatas = [
                {**meta, "embedding_type": "full", "dimensions": 768}
                for meta in metadatas[batch_idx:batch_end]
            ]

            self.collection_full.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings_full[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )

        print(f"✓ Full embeddings stored successfully!")

        # Step 4: Store PCA embeddings in ChromaDB
        print(f"\n{'='*70}")
        print(f"STEP 4: STORING PCA EMBEDDINGS (64D) IN CHROMADB...")
        print(f"{'='*70}")

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            # Add metadata indicating this is PCA embedding
            batch_metadatas = [
                {**meta, "embedding_type": "pca", "dimensions": self.pca_dimensions}
                for meta in metadatas[batch_idx:batch_end]
            ]

            self.collection_pca.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings_pca[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )

        print(f"✓ PCA embeddings stored successfully!")

        total_time = time.time() - start_time
        print(f"\n{'='*70}")
        print(f"✓ EMBEDDING GENERATION COMPLETE!")
        print(f"{'='*70}")
        print(f"Total embeddings stored: {len(ids)}")
        print(f"Full embeddings (768D): {len(ids)} in chroma_db_mean_content_full")
        print(f"PCA embeddings (64D): {len(ids)} in chroma_db_mean_content_PCA64")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Average time per article: {total_time/len(ids):.4f} seconds")
        print(f"Processing speed: {len(ids)/total_time:.2f} articles/second")

    def search_similar_articles(self, query_text: str, n_results: int = 5,
                               use_pca: bool = False) -> dict:
        """
        Search for similar articles using query text.

        Args:
            query_text: Query text to find similar articles
            n_results: Number of similar articles to return
            use_pca: If True, search in PCA collection; if False, search in full collection

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Apply PCA if needed
        if use_pca:
            if self.pca_model is None:
                # Try to load PCA model
                pca_model_path = self.chroma_db_path_pca / "pca_model.pkl"
                if pca_model_path.exists():
                    with open(pca_model_path, 'rb') as f:
                        self.pca_model = pickle.load(f)
                else:
                    raise ValueError("PCA model not found. Generate embeddings first.")

            query_embedding = self.pca_model.transform(query_embedding.reshape(1, -1))[0]
            collection = self.collection_pca
        else:
            collection = self.collection_full

        # Search in ChromaDB
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about both ChromaDB collections.

        Returns:
            Dictionary with collection information
        """
        count_full = self.collection_full.count()
        count_pca = self.collection_pca.count()

        return {
            "full_embeddings": {
                "total_embeddings": count_full,
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768,
                "db_path": str(self.chroma_db_path_full)
            },
            "pca_embeddings": {
                "total_embeddings": count_pca,
                "collection_name": self.collection_pca.name,
                "embedding_dimension": self.pca_dimensions,
                "db_path": str(self.chroma_db_path_pca),
                "explained_variance": f"{np.sum(self.pca_model.explained_variance_ratio_) * 100:.2f}%"
                    if self.pca_model else "N/A"
            },
            "pooling_method": "MEAN POOLING",
            "search_column": "content"
        }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS EMBEDDINGS WITH PCA DIMENSIONALITY REDUCTION")
    print("Full Embeddings: chroma_db_mean_content_full (768D)")
    print("PCA Embeddings: chroma_db_mean_content_PCA64 (64D)")
    print("="*70)
    print("\nLoading dataset...")

    df = pd.read_csv("final_cleaned_urdu_news.csv")

    print(f"\nDataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset info:")
    print(f"  - Total articles: {len(df)}")
    print(f"  - Unique categories: {df['Category'].nunique()}")
    print(f"  - Categories: {df['Category'].unique().tolist()}")

    # Show sample data
    print(f"\n{'='*70}")
    print("SAMPLE DATA PREVIEW")
    print(f"{'='*70}")
    sample = df.iloc[0]
    print(f"Headline: {sample['Headline'][:100]}...")
    print(f"Category: {sample['Category']}")
    print(f"Content preview: {sample['content'][:200]}...")
    print(f"Content length: {len(sample['content'])} characters")

    # Initialize embedding generator
    print(f"\n{'='*70}")
    print("INITIALIZING EMBEDDING GENERATOR")
    print(f"{'='*70}")

    embedder = UrduNewsEmbeddingGenerator(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path_full="./chroma_db_mean_content_full_claude",
        chroma_db_path_pca="./chroma_db_mean_content_PCA64_claude",
        pca_dimensions=64
    )

    # Generate embeddings and store in both ChromaDB collections
    embedder.generate_embeddings_for_dataset(
        df=df,
        content_column="content",
        headline_column="Headline",
        category_column="Category"
    )

    # Display collection statistics
    print("\n" + "="*70)
    print("CHROMADB COLLECTION STATISTICS")
    print("="*70)
    stats = embedder.get_collection_stats()
    print("\nFULL EMBEDDINGS COLLECTION:")
    for key, value in stats["full_embeddings"].items():
        print(f"  {key}: {value}")
    print("\nPCA EMBEDDINGS COLLECTION:")
    for key, value in stats["pca_embeddings"].items():
        print(f"  {key}: {value}")
    print(f"\nPooling method: {stats['pooling_method']}")
    print(f"Search column: {stats['search_column']}")

    # Test: Search in both collections
    print("\n" + "="*70)
    print("TESTING SEMANTIC SEARCH")
    print("="*70)

    query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"
    print(f"\nQuery: {query}\n")

    # Search in full embeddings
    print("="*70)
    print("RESULTS FROM FULL EMBEDDINGS (768D)")
    print("="*70)
    results_full = embedder.search_similar_articles(query_text=query, n_results=5, use_pca=False)

    for i, (doc_id, distance, metadata) in enumerate(zip(
        results_full['ids'][0],
        results_full['distances'][0],
        results_full['metadatas'][0]
    )):
        print(f"\nResult #{i+1}")
        print(f"  Similarity Score: {1 - distance:.4f}")
        print(f"  Headline: {metadata.get('headline', 'N/A')[:80]}...")
        print(f"  Category: {metadata.get('category', 'N/A')}")

    # Search in PCA embeddings
    print("\n" + "="*70)
    print("RESULTS FROM PCA EMBEDDINGS (64D)")
    print("="*70)
    results_pca = embedder.search_similar_articles(query_text=query, n_results=5, use_pca=True)

    for i, (doc_id, distance, metadata) in enumerate(zip(
        results_pca['ids'][0],
        results_pca['distances'][0],
        results_pca['metadatas'][0]
    )):
        print(f"\nResult #{i+1}")
        print(f"  Similarity Score: {1 - distance:.4f}")
        print(f"  Headline: {metadata.get('headline', 'N/A')[:80]}...")
        print(f"  Category: {metadata.get('category', 'N/A')}")

    print("\n" + "="*70)
    print("✓ PROCESSING COMPLETED SUCCESSFULLY!")
    print("✓ Two ChromaDB collections created:")
    print("  1. chroma_db_mean_content_full (768D)")
    print("  2. chroma_db_mean_content_PCA64 (64D)")
    print("="*70)

URDU NEWS EMBEDDINGS WITH PCA DIMENSIONALITY REDUCTION
Full Embeddings: chroma_db_mean_content_full (768D)
PCA Embeddings: chroma_db_mean_content_PCA64 (64D)

Loading dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 4
  - Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا...
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters

INITIALIZING EMBEDDING GENERATOR
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing ChromaDB for full embeddings at: chroma_db_mean_content_full_claude
I

## **Recommender Code PCA

In [1]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM WITH FULL VS PCA EMBEDDINGS COMPARISON
# Compares recommendations from 768D full embeddings and 64D PCA-reduced embeddings
# OUTPUTS RESULTS TO WORD DOCUMENT
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib for better visualization
rcParams['figure.figsize'] = (16, 10)
rcParams['font.size'] = 10
sns.set_style("whitegrid")

class UrduNewsRecommenderComparison:
    """
    Recommendation system that compares results from full embeddings (768D)
    and PCA-reduced embeddings (64D).
    Outputs results to Word document.
    """

    def __init__(self,
                 model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "chroma_db_mean_content_full_claude",
                 chroma_db_path_pca: str = "chroma_db_mean_content_PCA64_claude",
                 collection_name_full: str = "urdu_news_embeddings_full_mean_claude",
                 collection_name_pca: str = "urdu_news_embeddings_pca64_mean_claude",
                 output_doc_path: str = "Urdu_News_Full_vs_PCA64_Comparison_Report.docx"):
        """
        Initialize recommender with connections to both full and PCA databases.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path_full: Path to ChromaDB with full embeddings
            chroma_db_path_pca: Path to ChromaDB with PCA embeddings
            collection_name_full: Name of full embeddings collection
            collection_name_pca: Name of PCA embeddings collection
            output_doc_path: Path for the output Word document
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_pca = Path(chroma_db_path_pca)
        self.pca_model = None
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Load PCA model
        pca_model_path = self.chroma_db_path_pca / "pca_model.pkl"
        if pca_model_path.exists():
            print(f"Loading PCA model from: {pca_model_path}")
            self.add_paragraph(f"Loading PCA model from: {pca_model_path}")
            with open(pca_model_path, 'rb') as f:
                self.pca_model = pickle.load(f)
            print(f"✓ PCA model loaded (64 dimensions)")
            self.add_paragraph(f"✓ PCA model loaded (64 dimensions)")
        else:
            raise FileNotFoundError(f"PCA model not found at {pca_model_path}")

        # Connect to full embeddings ChromaDB
        print(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.add_paragraph(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.client_full = chromadb.PersistentClient(path=str(self.chroma_db_path_full))

        try:
            self.collection_full = self.client_full.get_collection(name=collection_name_full)
            print(f"✓ Connected to full embeddings collection: {collection_name_full}")
            print(f"✓ Total articles in full database: {self.collection_full.count()}")
            self.add_paragraph(f"✓ Connected to full embeddings collection: {collection_name_full}")
            self.add_paragraph(f"✓ Total articles in full database: {self.collection_full.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_full}'")
            raise e

        # Connect to PCA embeddings ChromaDB
        print(f"Connecting to PCA Embeddings ChromaDB at: {self.chroma_db_path_pca}")
        self.add_paragraph(f"Connecting to PCA Embeddings ChromaDB at: {self.chroma_db_path_pca}")
        self.client_pca = chromadb.PersistentClient(path=str(self.chroma_db_path_pca))

        try:
            self.collection_pca = self.client_pca.get_collection(name=collection_name_pca)
            print(f"✓ Connected to PCA embeddings collection: {collection_name_pca}")
            print(f"✓ Total articles in PCA database: {self.collection_pca.count()}")
            self.add_paragraph(f"✓ Connected to PCA embeddings collection: {collection_name_pca}")
            self.add_paragraph(f"✓ Total articles in PCA database: {self.collection_pca.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_pca}'")
            raise e

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System\nFull (768D) vs PCA (64D) Comparison Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text='', style=None):
        """Add paragraph to document."""
        para = self.doc.add_paragraph(text, style)
        return para

    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'

        for row_idx, row_data in enumerate(data):
            if headers and row_idx == 0:
                continue
            if not headers:
                cells = table.rows[row_idx].cells
            else:
                cells = table.add_row().cells
            for col_idx, cell_data in enumerate(row_data):
                cells[col_idx].text = str(cell_data)

        return table

    def add_image(self, image_path, width_inches=6):
        """Add image to document."""
        try:
            self.doc.add_picture(image_path, width=Inches(width_inches))
            last_paragraph = self.doc.paragraphs[-1]
            last_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
        except Exception as e:
            self.add_paragraph(f"Error adding image: {str(e)}")

    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"Word document saved to: {self.output_doc_path}")

    def mean_pooling(self, model_output, attention_mask):
        """Apply MEAN POOLING to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50, apply_pca: bool = False) -> np.ndarray:
        """
        Generate embedding for query text using MEAN POOLING.
        Optionally applies PCA dimensionality reduction.
        """
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # Process short queries
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embedding = embeddings.cpu().detach().numpy()[0]

            # Apply PCA if requested
            if apply_pca and self.pca_model is not None:
                embedding = self.pca_model.transform(embedding.reshape(1, -1))[0]

            return embedding

        # For long queries: use chunking
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)

        # Apply PCA if requested
        if apply_pca and self.pca_model is not None:
            final_embedding = self.pca_model.transform(final_embedding.reshape(1, -1))[0]

        return final_embedding

    def get_recommendations_comparison(self, query: str, n_results: int = 10,
                                      filter_category: str = None) -> dict:
        """
        Get recommendations from both full and PCA embeddings for comparison.
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS - FULL vs PCA COMPARISON")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Generate embeddings for both spaces
        print("→ Generating query embedding (768D - Full)...")
        query_embedding_full = self.generate_query_embedding(query, apply_pca=False)
        print(f"  ✓ Full embedding generated: shape {query_embedding_full.shape}")

        print("→ Generating query embedding (64D - PCA)...")
        query_embedding_pca = self.generate_query_embedding(query, apply_pca=True)
        print(f"  ✓ PCA embedding generated: shape {query_embedding_pca.shape}")

        # Search in full embeddings
        print("\n→ Searching in Full Embeddings (768D)...")
        results_full = self.collection_full.query(
            query_embeddings=[query_embedding_full.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results_full['ids'][0])} recommendations")

        # Search in PCA embeddings
        print("→ Searching in PCA Embeddings (64D)...")
        results_pca = self.collection_pca.query(
            query_embeddings=[query_embedding_pca.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results_pca['ids'][0])} recommendations\n")

        return {
            'full': results_full,
            'pca': results_pca,
            'query': query,
            'n_results': n_results
        }

    def compare_and_visualize(self, comparison_results: dict, df: pd.DataFrame = None, query_number: int = 1):
        """
        Compare and visualize recommendations from full vs PCA embeddings.
        Adds results to Word document.
        """
        results_full = comparison_results['full']
        results_pca = comparison_results['pca']
        query = comparison_results['query']
        n_results = comparison_results.get('n_results', 10)

        if not results_full['ids'] or not results_pca['ids']:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return

        # Add query heading to document
        self.add_heading(f"Query {query_number}: Comparison Results", level=1)
        self.add_paragraph(f"Query: {query[:200]}...")
        self.add_paragraph(f"Top {n_results} results from each method")
        self.add_paragraph('')

        # Extract data
        full_ids = results_full['ids'][0]
        full_distances = results_full['distances'][0]
        full_metadatas = results_full['metadatas'][0]

        pca_ids = results_pca['ids'][0]
        pca_distances = results_pca['distances'][0]
        pca_metadatas = results_pca['metadatas'][0]

        # Convert distances to similarity scores
        full_similarities = [1 - d for d in full_distances]
        pca_similarities = [1 - d for d in pca_distances]

        # Track overlap between results
        full_set = set(full_ids)
        pca_set = set(pca_ids)
        overlap = full_set.intersection(pca_set)

        # Print and add to doc: comparison summary
        print(f"\n{'='*70}")
        print(f"RECOMMENDATION COMPARISON SUMMARY - Query {query_number}")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...\n")
        print(f"Total recommendations per method: {n_results}")
        print(f"Articles in common: {len(overlap)} ({len(overlap)/n_results*100:.1f}%)")
        print(f"Unique to Full (768D): {len(full_set - pca_set)}")
        print(f"Unique to PCA (64D): {len(pca_set - full_set)}")

        self.add_heading("Comparison Summary", level=2)
        summary_data = [
            ["Total recommendations per method", n_results],
            ["Articles in common", f"{len(overlap)} ({len(overlap)/n_results*100:.1f}%)"],
            ["Unique to Full (768D)", len(full_set - pca_set)],
            ["Unique to PCA (64D)", len(pca_set - full_set)]
        ]
        self.add_table(summary_data, headers=["Metric", "Value"])
        self.add_paragraph('')

        # Create side-by-side comparison table for document
        self.add_heading("Side-by-Side Comparison", level=2)
        comparison_table_data = []

        for i in range(n_results):
            rank = i + 1

            # Full embedding info
            if i < len(full_ids):
                full_headline = full_metadatas[i].get('headline', 'N/A')[:80]
                full_sim = f"{full_similarities[i]:.4f}"
            else:
                full_headline = "N/A"
                full_sim = "0.0000"

            # PCA embedding info
            if i < len(pca_ids):
                pca_headline = pca_metadatas[i].get('headline', 'N/A')[:80]
                pca_sim = f"{pca_similarities[i]:.4f}"
            else:
                pca_headline = "N/A"
                pca_sim = "0.0000"

            comparison_table_data.append([rank, full_headline, full_sim, pca_headline, pca_sim])

        self.add_table(comparison_table_data,
                      headers=["Rank", "Full (768D) Headline", "Full Similarity",
                              "PCA (64D) Headline", "PCA Similarity"])
        self.add_paragraph('')

        # Create and add visualization
        image_path = self._create_visualizations(overlap, full_set, pca_set, query, n_results, query_number)

        self.add_heading("Overlap Visualization", level=2)
        if image_path:
            self.add_image(image_path, width_inches=5)
        self.add_paragraph('')

    def _create_visualizations(self, overlap, full_set, pca_set, query, n_results, query_number):
        """Create comparison visualizations and return image path."""
        fig = plt.figure(figsize=(8, 6))

        # Overlap Analysis
        ax = plt.subplot(1, 1, 1)

        overlap_count = len(overlap)
        full_only = len(full_set - pca_set)
        pca_only = len(pca_set - full_set)

        sizes = [overlap_count, full_only, pca_only]
        labels = [f'Both\n({overlap_count})', f'Full Only\n({full_only})', f'PCA Only\n({pca_only})']
        colors_pie = ['#4CAF50', '#2E86AB', '#A23B72']
        explode = (0.1, 0, 0)

        ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 10})
        ax.set_title('Recommendation Overlap Full Vs PCA',
                     fontsize=12, fontweight='bold', pad=15)

        # Overall title
        fig.suptitle(f'Query {query_number} - Recommendation Overlap\nTop {n_results} Results',
                    fontsize=14, fontweight='bold', y=0.995)

        plt.tight_layout(rect=[0, 0.03, 1, 0.985])

        image_path = f'recommendation_comparison_query_{query_number}.png'
        plt.savefig(image_path, dpi=300, bbox_inches='tight')
        print(f"\n✓ Visualization saved as '{image_path}'")
        plt.close()

        return image_path

    def get_statistics(self) -> dict:
        """Get statistics about both recommendation systems."""
        explained_var = np.sum(self.pca_model.explained_variance_ratio_) * 100 if self.pca_model else 0

        return {
            "model": self.model_name,
            "device": str(self.device),
            "full_embeddings": {
                "total_articles": self.collection_full.count(),
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768
            },
            "pca_embeddings": {
                "total_articles": self.collection_pca.count(),
                "collection_name": self.collection_pca.name,
                "embedding_dimension": 64,
                "explained_variance": f"{explained_var:.2f}%"
            },
            "pooling_method": "MEAN POOLING"
        }

    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        stats = self.get_statistics()

        stats_data = [
            ["Model", stats['model']],
            ["Device", stats['device']],
            ["Pooling Method", stats['pooling_method']]
        ]
        self.add_table(stats_data, headers=["Parameter", "Value"])

        self.add_heading("Full Embeddings (768D)", level=2)
        full_stats = [[k, v] for k, v in stats['full_embeddings'].items()]
        self.add_table(full_stats, headers=["Metric", "Value"])

        self.add_heading("PCA Embeddings (64D)", level=2)
        pca_stats = [[k, v] for k, v in stats['pca_embeddings'].items()]
        self.add_table(pca_stats, headers=["Metric", "Value"])
        self.add_paragraph('')


# =============================================================================
# MAIN EXECUTION - COMPARISON SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION COMPARISON SYSTEM")
    print("Full Embeddings (768D) vs PCA Embeddings (64D)")
    print("OUTPUTTING TO WORD DOCUMENT")
    print("="*70)

    # Load dataset
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender
    print("\n" + "="*70)
    print("INITIALIZING COMPARISON SYSTEM")
    print("="*70)

    recommender = UrduNewsRecommenderComparison(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path_full="chroma_db_mean_content_full_claude",
        chroma_db_path_pca="chroma_db_mean_content_PCA64_claude",
        collection_name_full="urdu_news_embeddings_full_mean_claude",
        collection_name_pca="urdu_news_embeddings_pca64_mean_claude",
        output_doc_path="Urdu_News_Full_vs_PCA64_Comparison_Report.docx"
    )

    # Display and add system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    print(f"Model: {stats['model']}")
    print(f"Device: {stats['device']}")
    print(f"Pooling Method: {stats['pooling_method']}")
    print(f"\nFull Embeddings (768D):")
    for key, value in stats['full_embeddings'].items():
        print(f"  {key}: {value}")
    print(f"\nPCA Embeddings (64D):")
    for key, value in stats['pca_embeddings'].items():
        print(f"  {key}: {value}")

    recommender.add_statistics_to_doc()

    # Define all queries
    queries = [
    "ریاضی اس سوال جواب دے پہلی نظر ریاضی سان سوال لگتا مگر اس انٹرنیٹ متعدد افراد ذہنوں پریشان کرکے رکھ ہےکیا اپنی ریاضی صلاحیت پورا بھروسا ہاں اس سوال جواب جس فیس بک صارفین ذہنوں گھما رکھ جس وجہ اس دو ممکنہ جواب ہیںاسے گو تھمبل تیار کیا اس فیس بک لوگوں ",
    "پاکستان اسٹاک ایکسچینج ملا جلا رجحان ہنڈرڈ انڈیکس پوائنٹس کمی بند پاکستان اسٹاک ایکسچینج اج کاروبار اختتام ملا جلا رجحان دیکھا گیا پاکستان اسٹاک ایکسچینج اج کاروبار اغاز اتار چڑھا جاری ملا جلا رجحان دیکھنے ایا اج کاروبار دوران کمپنیوں کروڑ شیئرز سودے",
    "سام سنگ نئے فلیگ شپ فون تاریخ رونمائی سامنے گئی سام سنگ اپنے نئے فلیگ شپ فون گلیکسی نوٹ ئندہ ماہ متعارف کرانے باضابطہ اعلان کردیا ہےسام سنگ جانب اگست گلیکسی نوٹ متعارف کرایا جائے جس گلیکسی زی فولڈ گلیکسی زی فلپ جی ورژن ایک لائن ایونٹ پیش کیے جانے امک",
    "سلمان خان دوستوں دوستی لے ڈوبی ممبئی ویب ڈیسک بالی وڈ پنڈتوں کہنا بظاہر نظر پروڈکشن میدان اترنے سلمان خان پہلی بار جھٹکا لگنے اپنے دوستوں بچوں عطیہ شیٹی سورج پنچولی ان فلم ہیرو ریلیز ہو چکی اس رپورٹ اچھی نہیں تفصیلات سلمان خان فلم ہیرو اداکارہ کردار ",
    "ئی فون متعارف کرانے تاریخ سامنے گئی رواں سال شروع ایسی اطلاعات سامنے ئی تھیں ئی فون ایٹ اس بار معمول ستمبر سامنے نہیں سکے بلکہ اسے پیش کیا جائے گاتاہم ایپل اس نئے مکمل ری ڈیزائن اسمارٹ فون تاریخ رونمائی سامنے نہیں ئی مگر پتا چل گیا ستمبر عشرے متعارف ",
    "پاکستان ویسٹ انڈیز قسمت بدلنے خواہاں برج ٹان پاکستان عالمی چیمپیئن ویسٹ انڈیز درمیان چار ٹی میچوں سیریز اج اغاز ہونے جا جہاں میزبان ٹیم متحدہ عرب امارات گرین شرٹس بدترین کارکردگی بھلا بہترین کھیل پیش کرنے کوشش کرے گیگزشتہ سال ستمبر ان تین میچوں بے دا",
    "ایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم کراچیایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم ہوگیا کوارٹر فائنل شکست بعد محمد سجاد ابو صائم ایونٹ ہوگئے کراچی جاری ایونٹ کوارٹر فائنل ایران عامر سرکوش مقابلہ پاکستان محمد سجاد ایرانی کیوسٹ عمدہ کارکرد",
    "یو ایس اوپن ٹینس نوواک جوکووچ تیسرے رانڈ پہنچ گئے نیویارک اگست یو ایس اوپن ٹاپ سیڈ دفاعی چیمپئن نوواک جوکووچ باسانی تیسرے رانڈ پہنچ گئے اکہترویں فتح پیٹ سمپراس ہم پلہ بن گئے سرینا ولیمز میڈیسن کیز اپنے میچز سرخرو رہیں نیویارک ٹینس ایرینا جاری سیزن کا",
    "پی ٹی اے زونگ جی اشتہارات واپس لینے ہدایت پاکستان ٹیلی کمیونیکشن اتھارٹی پی ٹی اے چائنا موبائل پاکستان سی پی ایم یعنی زونگ جی ٹیکنالوجی اشتہارات تشہیر روکنے حکم ہےپی ٹی اے جانب جاری بیان زونگ جانب میڈیا حالیہ جی اشتہارات عوام گاہ کیا جاتا کمپنی فائیو",
    "ناقدین نظر بہترین فلمیں ناقدین نظر بہترین فلمیں امریکن گریفیٹی سائیکو سم لائیک اٹ ہاٹ منتھس ویکس اینڈ ڈیز گون ود ونڈ مائی لیفٹ فٹ ہوپ ڈریمز پینز لیبیرنتھ مون لائٹ سنگنگ ان رین تھری کلرز ریڈ بوائے ہڈ کاسا بلانکا گاڈ فادر سیٹزن کین ناقدین نظر بہترین فل",
    "پاکستانی اسکواڈ کرائسٹ چرچ کوئنز ٹان روانہ ہو لاہور دسمبر نیوزی لینڈ موجود قومی کرکٹ اسکواڈ مینیجڈ ئسولیشن چھوڑنے اجازت مل گئی قومی اسکواڈ شامل اکاون ارکان کرائسٹ چرچ مینیجڈ ئسولیشن چھوڑ دو ارکان اپنی ئسولیشن مدت پوری کریں نیوزی لینڈ موجود قومی اسکوا",
    "سرچ ہونے ڈیوائسز پسند ہو نہ ہو مگر ایپل ئی فون سکس ایس گوگل مقبول ڈیوائس قرار دے ہےگوگل جانب ٹرینڈز ٹیکنالوجی فہرست ئی فون سکس ایس سرچ ہونے باعث سرفہرست رہاویسے حیرت انگیز نہیں ئی فون نئے ماڈلز لگ بھگ سال بہترین قرار پاتے لوگوں نئے ئی فون جاننے تجسس ",
    "پاکستان اسٹاک ایکسچینج پوائنٹس اضافہ کراچی اپریل کاروباری ہفتے تیسرے روز پاکستان اسٹاک ایکسچینج مثبت رجحان انڈیکس پوائنٹس بہتری بدھ پاکستان اسٹاک ایکسچینج بہتری رجحان ایس ای انڈیکس پوائنٹس اضافے بعد ہزار پوائنٹس سطح پہنچ گیا کاروبار دوران کروڑ لاکھ ہ",
     ]

    # Execute all queries
    for idx, query in enumerate(queries, 1):
        print("\n\n" + "="*70)
        print(f"EXECUTING QUERY {idx} OF {len(queries)}")
        print("="*70)

        results = recommender.get_recommendations_comparison(
            query=query,
            n_results=50
        )

        recommender.compare_and_visualize(results, df=df, query_number=idx)

    # Save the final document
    recommender.save_document()

    print("\n" + "="*70)
    print("✓ COMPARISON SYSTEM DEMO COMPLETED!")
    print("✓ Full embeddings: 768 dimensions")
    print("✓ PCA embeddings: 64 dimensions")
    print(f"✓ All {len(queries)} queries executed")
    print(f"✓ Comprehensive report saved to: {recommender.output_doc_path}")
    print("✓ Visualizations saved as PNG files")
    print("="*70)

URDU NEWS RECOMMENDATION COMPARISON SYSTEM
Full Embeddings (768D) vs PCA Embeddings (64D)
OUTPUTTING TO WORD DOCUMENT

Loading dataset for content display...
✓ Dataset loaded: 111853 articles

INITIALIZING COMPARISON SYSTEM
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Loading PCA model from: chroma_db_mean_content_PCA64_claude/pca_model.pkl
✓ PCA model loaded (64 dimensions)
Connecting to Full Embeddings ChromaDB at: chroma_db_mean_content_full_claude
✓ Connected to full embeddings collection: urdu_news_embeddings_full_mean_claude
✓ Total articles in full database: 111853
Connecting to PCA Embeddings ChromaDB at: chroma_db_mean_content_PCA64_claude
✓ Connected to PCA embeddings collection: urdu_news_embeddings_pca64_mean_claude
✓ Total articles in PCA database: 111853

SYSTEM STATISTICS
Model: urduhack/roberta-urdu-small
Device: cuda
Pooling Method: MEAN POOLING

Full Embeddings (768D):
  total_articles: 111853
  collection_name: urdu_news_embeddings_full_mean_claude
 

In [1]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM WITH FULL VS PCA EMBEDDINGS COMPARISON
# Compares recommendations from 768D full embeddings and 64D PCA-reduced embeddings
# OUTPUTS RESULTS TO WORD DOCUMENT AND DISPLAYS ON SCREEN WITH DATASET INDICES
# =============================================================================
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
# Configure matplotlib for better visualization
rcParams['figure.figsize'] = (16, 10)
rcParams['font.size'] = 10
sns.set_style("whitegrid")
class UrduNewsRecommenderComparison:
    """
    Recommendation system that compares results from full embeddings (768D)
    and PCA-reduced embeddings (64D).
    Outputs results to Word document and displays on screen with dataset indices.
    """
    def __init__(self,
                 model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "chroma_db_mean_content_full_claude",
                 chroma_db_path_pca: str = "chroma_db_mean_content_PCA64_claude",
                 collection_name_full: str = "urdu_news_embeddings_full_mean_claude",
                 collection_name_pca: str = "urdu_news_embeddings_pca64_mean_claude",
                 output_doc_path: str = "Urdu_News_Full_vs_PCA64_Comparison_Report.docx"):
        """
        Initialize recommender with connections to both full and PCA databases.
        Args:
            model_name: HuggingFace model identifier
            chroma_db_path_full: Path to ChromaDB with full embeddings
            chroma_db_path_pca: Path to ChromaDB with PCA embeddings
            collection_name_full: Name of full embeddings collection
            collection_name_pca: Name of PCA embeddings collection
            output_doc_path: Path for the output Word document
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_pca = Path(chroma_db_path_pca)
        self.pca_model = None
        self.output_doc_path = output_doc_path
        # Initialize Word document
        self.doc = Document()
        self.setup_document()
        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)
        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()
        # Load PCA model
        pca_model_path = self.chroma_db_path_pca / "pca_model.pkl"
        if pca_model_path.exists():
            print(f"Loading PCA model from: {pca_model_path}")
            self.add_paragraph(f"Loading PCA model from: {pca_model_path}")
            with open(pca_model_path, 'rb') as f:
                self.pca_model = pickle.load(f)
            print(f"✓ PCA model loaded (64 dimensions)")
            self.add_paragraph(f"✓ PCA model loaded (64 dimensions)")
        else:
            raise FileNotFoundError(f"PCA model not found at {pca_model_path}")
        # Connect to full embeddings ChromaDB
        print(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.add_paragraph(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.client_full = chromadb.PersistentClient(path=str(self.chroma_db_path_full))
        try:
            self.collection_full = self.client_full.get_collection(name=collection_name_full)
            print(f"✓ Connected to full embeddings collection: {collection_name_full}")
            print(f"✓ Total articles in full database: {self.collection_full.count()}")
            self.add_paragraph(f"✓ Connected to full embeddings collection: {collection_name_full}")
            self.add_paragraph(f"✓ Total articles in full database: {self.collection_full.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_full}'")
            raise e
        # Connect to PCA embeddings ChromaDB
        print(f"Connecting to PCA Embeddings ChromaDB at: {self.chroma_db_path_pca}")
        self.add_paragraph(f"Connecting to PCA Embeddings ChromaDB at: {self.chroma_db_path_pca}")
        self.client_pca = chromadb.PersistentClient(path=str(self.chroma_db_path_pca))
        try:
            self.collection_pca = self.client_pca.get_collection(name=collection_name_pca)
            print(f"✓ Connected to PCA embeddings collection: {collection_name_pca}")
            print(f"✓ Total articles in PCA database: {self.collection_pca.count()}")
            self.add_paragraph(f"✓ Connected to PCA embeddings collection: {collection_name_pca}")
            self.add_paragraph(f"✓ Total articles in PCA database: {self.collection_pca.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_pca}'")
            raise e
    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System\nFull (768D) vs PCA (64D) Comparison Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        # Add empty line
        self.doc.add_paragraph()
    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading
    def add_paragraph(self, text='', style=None):
        """Add paragraph to document."""
        para = self.doc.add_paragraph(text, style)
        return para
    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'
        for row_idx, row_data in enumerate(data):
            if headers and row_idx == 0:
                continue
            if not headers:
                cells = table.rows[row_idx].cells
            else:
                cells = table.add_row().cells
            for col_idx, cell_data in enumerate(row_data):
                cells[col_idx].text = str(cell_data)
        return table
    def add_image(self, image_path, width_inches=6):
        """Add image to document."""
        try:
            self.doc.add_picture(image_path, width=Inches(width_inches))
            last_paragraph = self.doc.paragraphs[-1]
            last_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
        except Exception as e:
            self.add_paragraph(f"Error adding image: {str(e)}")
    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"Word document saved to: {self.output_doc_path}")
    def mean_pooling(self, model_output, attention_mask):
        """Apply MEAN POOLING to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings
    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50, apply_pca: bool = False) -> np.ndarray:
        """
        Generate embedding for query text using MEAN POOLING.
        Optionally applies PCA dimensionality reduction.
        """
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)
        # Process short queries
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}
            with torch.no_grad():
                model_output = self.model(**encoded_input)
            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embedding = embeddings.cpu().detach().numpy()[0]
            # Apply PCA if requested
            if apply_pca and self.pca_model is not None:
                embedding = self.pca_model.transform(embedding.reshape(1, -1))[0]
            return embedding
        # For long queries: use chunking
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []
        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break
            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}
            with torch.no_grad():
                model_output = self.model(**encoded_input)
            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])
        final_embedding = np.mean(chunk_embeddings, axis=0)
        # Apply PCA if requested
        if apply_pca and self.pca_model is not None:
            final_embedding = self.pca_model.transform(final_embedding.reshape(1, -1))[0]
        return final_embedding
    def get_recommendations_comparison(self, query: str, n_results: int = 10,
                                      filter_category: str = None) -> dict:
        """
        Get recommendations from both full and PCA embeddings for comparison.
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS - FULL vs PCA COMPARISON")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")
        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}
        # Generate embeddings for both spaces
        print("→ Generating query embedding (768D - Full)...")
        query_embedding_full = self.generate_query_embedding(query, apply_pca=False)
        print(f" ✓ Full embedding generated: shape {query_embedding_full.shape}")
        print("→ Generating query embedding (64D - PCA)...")
        query_embedding_pca = self.generate_query_embedding(query, apply_pca=True)
        print(f" ✓ PCA embedding generated: shape {query_embedding_pca.shape}")
        # Search in full embeddings
        print("\n→ Searching in Full Embeddings (768D)...")
        results_full = self.collection_full.query(
            query_embeddings=[query_embedding_full.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f" ✓ Found {len(results_full['ids'][0])} recommendations")
        # Search in PCA embeddings
        print("→ Searching in PCA Embeddings (64D)...")
        results_pca = self.collection_pca.query(
            query_embeddings=[query_embedding_pca.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f" ✓ Found {len(results_pca['ids'][0])} recommendations\n")
        return {
            'full': results_full,
            'pca': results_pca,
            'query': query,
            'n_results': n_results
        }
    def compare_and_visualize(self, comparison_results: dict, df: pd.DataFrame = None, query_number: int = 1):
        """
        Compare and visualize recommendations from full vs PCA embeddings.
        Adds results to Word document.
        Displays recommendations on screen with dataset indices.
        """
        results_full = comparison_results['full']
        results_pca = comparison_results['pca']
        query = comparison_results['query']
        n_results = comparison_results.get('n_results', 10)
        if not results_full['ids'] or not results_pca['ids']:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return
        # Add query heading to document
        self.add_heading(f"Query {query_number}: Comparison Results", level=1)
        self.add_paragraph(f"Query: {query[:200]}...")
        self.add_paragraph(f"Top {n_results} results from each method")
        self.add_paragraph('')
        # Extract data
        full_ids = results_full['ids'][0]
        full_distances = results_full['distances'][0]
        full_metadatas = results_full['metadatas'][0]
        pca_ids = results_pca['ids'][0]
        pca_distances = results_pca['distances'][0]
        pca_metadatas = results_pca['metadatas'][0]
        # Convert distances to similarity scores
        full_similarities = [1 - d for d in full_distances]
        pca_similarities = [1 - d for d in pca_distances]
        # Track overlap between results
        full_set = set(full_ids)
        pca_set = set(pca_ids)
        overlap = full_set.intersection(pca_set)
        # Print and add to doc: comparison summary
        print(f"\n{'='*70}")
        print(f"RECOMMENDATION COMPARISON SUMMARY - Query {query_number}")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...\n")
        print(f"Total recommendations per method: {n_results}")
        print(f"Articles in common: {len(overlap)} ({len(overlap)/n_results*100:.1f}%)")
        print(f"Unique to Full (768D): {len(full_set - pca_set)}")
        print(f"Unique to PCA (64D): {len(pca_set - full_set)}")
        self.add_heading("Comparison Summary", level=2)
        summary_data = [
            ["Total recommendations per method", n_results],
            ["Articles in common", f"{len(overlap)} ({len(overlap)/n_results*100:.1f}%)"],
            ["Unique to Full (768D)", len(full_set - pca_set)],
            ["Unique to PCA (64D)", len(pca_set - full_set)]
        ]
        self.add_table(summary_data, headers=["Metric", "Value"])
        self.add_paragraph('')
        # Display recommendations on screen
        print(f"\n{'-'*50}")
        print("Full Embeddings (768D) Recommendations:")
        print(f"{'-'*50}")
        for i in range(len(full_ids)):
            id_ = full_ids[i]
            try:
                dataset_index = int(id_)  # Assuming ID is string of dataset index
            except ValueError:
                dataset_index = "N/A"  # If not convertible
            headline = full_metadatas[i].get('headline', 'N/A')[:80]
            sim = full_similarities[i]
            print(f"Rank {i+1}: Dataset Index {dataset_index}, Headline: {headline}, Similarity: {sim:.4f}")
        print(f"\n{'-'*50}")
        print("PCA Embeddings (64D) Recommendations:")
        print(f"{'-'*50}")
        for i in range(len(pca_ids)):
            id_ = pca_ids[i]
            try:
                dataset_index = int(id_)  # Assuming ID is string of dataset index
            except ValueError:
                dataset_index = "N/A"  # If not convertible
            headline = pca_metadatas[i].get('headline', 'N/A')[:80]
            sim = pca_similarities[i]
            print(f"Rank {i+1}: Dataset Index {dataset_index}, Headline: {headline}, Similarity: {sim:.4f}")
        # Create side-by-side comparison table for document (including dataset indices)
        self.add_heading("Side-by-Side Comparison", level=2)
        comparison_table_data = []
        for i in range(n_results):
            rank = i + 1
            # Full embedding info
            if i < len(full_ids):
                full_id = full_ids[i]
                try:
                    full_index = int(full_id)
                except ValueError:
                    full_index = "N/A"
                full_headline = full_metadatas[i].get('headline', 'N/A')[:80]
                full_sim = f"{full_similarities[i]:.4f}"
            else:
                full_index = "N/A"
                full_headline = "N/A"
                full_sim = "0.0000"
            # PCA embedding info
            if i < len(pca_ids):
                pca_id = pca_ids[i]
                try:
                    pca_index = int(pca_id)
                except ValueError:
                    pca_index = "N/A"
                pca_headline = pca_metadatas[i].get('headline', 'N/A')[:80]
                pca_sim = f"{pca_similarities[i]:.4f}"
            else:
                pca_index = "N/A"
                pca_headline = "N/A"
                pca_sim = "0.0000"
            comparison_table_data.append([rank, full_index, full_headline, full_sim, pca_index, pca_headline, pca_sim])
        self.add_table(comparison_table_data,
                      headers=["Rank", "Full (768D) Dataset Index", "Full (768D) Headline", "Full Similarity",
                              "PCA (64D) Dataset Index", "PCA (64D) Headline", "PCA Similarity"])
        self.add_paragraph('')
        # Create and add visualization
        image_path = self._create_visualizations(overlap, full_set, pca_set, query, n_results, query_number)
        self.add_heading("Overlap Visualization", level=2)
        if image_path:
            self.add_image(image_path, width_inches=5)
        self.add_paragraph('')
    def _create_visualizations(self, overlap, full_set, pca_set, query, n_results, query_number):
        """Create comparison visualizations and return image path."""
        fig = plt.figure(figsize=(8, 6))
        # Overlap Analysis
        ax = plt.subplot(1, 1, 1)
        overlap_count = len(overlap)
        full_only = len(full_set - pca_set)
        pca_only = len(pca_set - full_set)
        sizes = [overlap_count, full_only, pca_only]
        labels = [f'Both\n({overlap_count})', f'Full Only\n({full_only})', f'PCA Only\n({pca_only})']
        colors_pie = ['#4CAF50', '#2E86AB', '#A23B72']
        explode = (0.1, 0, 0)
        ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 10})
        ax.set_title('Recommendation Overlap Full Vs PCA',
                     fontsize=12, fontweight='bold', pad=15)
        # Overall title
        fig.suptitle(f'Query {query_number} - Recommendation Overlap\nTop {n_results} Results',
                    fontsize=14, fontweight='bold', y=0.995)
        plt.tight_layout(rect=[0, 0.03, 1, 0.985])
        image_path = f'recommendation_comparison_query_{query_number}.png'
        plt.savefig(image_path, dpi=300, bbox_inches='tight')
        print(f"\n✓ Visualization saved as '{image_path}'")
        plt.close()
        return image_path
    def get_statistics(self) -> dict:
        """Get statistics about both recommendation systems."""
        explained_var = np.sum(self.pca_model.explained_variance_ratio_) * 100 if self.pca_model else 0
        return {
            "model": self.model_name,
            "device": str(self.device),
            "full_embeddings": {
                "total_articles": self.collection_full.count(),
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768
            },
            "pca_embeddings": {
                "total_articles": self.collection_pca.count(),
                "collection_name": self.collection_pca.name,
                "embedding_dimension": 64,
                "explained_variance": f"{explained_var:.2f}%"
            },
            "pooling_method": "MEAN POOLING"
        }
    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        stats = self.get_statistics()
        stats_data = [
            ["Model", stats['model']],
            ["Device", stats['device']],
            ["Pooling Method", stats['pooling_method']]
        ]
        self.add_table(stats_data, headers=["Parameter", "Value"])
        self.add_heading("Full Embeddings (768D)", level=2)
        full_stats = [[k, v] for k, v in stats['full_embeddings'].items()]
        self.add_table(full_stats, headers=["Metric", "Value"])
        self.add_heading("PCA Embeddings (64D)", level=2)
        pca_stats = [[k, v] for k, v in stats['pca_embeddings'].items()]
        self.add_table(pca_stats, headers=["Metric", "Value"])
        self.add_paragraph('')
# =============================================================================
# MAIN EXECUTION - COMPARISON SYSTEM
# =============================================================================
if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION COMPARISON SYSTEM")
    print("Full Embeddings (768D) vs PCA Embeddings (64D)")
    print("OUTPUTTING TO WORD DOCUMENT")
    print("="*70)
    # Load dataset
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    print(f"✓ Dataset loaded: {len(df)} articles")
    # Initialize recommender
    print("\n" + "="*70)
    print("INITIALIZING COMPARISON SYSTEM")
    print("="*70)
    recommender = UrduNewsRecommenderComparison(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path_full="chroma_db_mean_content_full_claude",
        chroma_db_path_pca="chroma_db_mean_content_PCA64_claude",
        collection_name_full="urdu_news_embeddings_full_mean_claude",
        collection_name_pca="urdu_news_embeddings_pca64_mean_claude",
        output_doc_path="Urdu_News_Full_vs_PCA64_Comparison_Report.docx"
    )
    # Display and add system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    print(f"Model: {stats['model']}")
    print(f"Device: {stats['device']}")
    print(f"Pooling Method: {stats['pooling_method']}")
    print(f"\nFull Embeddings (768D):")
    for key, value in stats['full_embeddings'].items():
        print(f" {key}: {value}")
    print(f"\nPCA Embeddings (64D):")
    for key, value in stats['pca_embeddings'].items():
        print(f" {key}: {value}")
    recommender.add_statistics_to_doc()
    # Define all queries
    queries = [
    "ریاضی اس سوال جواب دے پہلی نظر ریاضی سان سوال لگتا مگر اس انٹرنیٹ متعدد افراد ذہنوں پریشان کرکے رکھ ہےکیا اپنی ریاضی صلاحیت پورا بھروسا ہاں اس سوال جواب جس فیس بک صارفین ذہنوں گھما رکھ جس وجہ اس دو ممکنہ جواب ہیںاسے گو تھمبل تیار کیا اس فیس بک لوگوں ",
    "پاکستان اسٹاک ایکسچینج ملا جلا رجحان ہنڈرڈ انڈیکس پوائنٹس کمی بند پاکستان اسٹاک ایکسچینج اج کاروبار اختتام ملا جلا رجحان دیکھا گیا پاکستان اسٹاک ایکسچینج اج کاروبار اغاز اتار چڑھا جاری ملا جلا رجحان دیکھنے ایا اج کاروبار دوران کمپنیوں کروڑ شیئرز سودے",
    "سام سنگ نئے فلیگ شپ فون تاریخ رونمائی سامنے گئی سام سنگ اپنے نئے فلیگ شپ فون گلیکسی نوٹ ئندہ ماہ متعارف کرانے باضابطہ اعلان کردیا ہےسام سنگ جانب اگست گلیکسی نوٹ متعارف کرایا جائے جس گلیکسی زی فولڈ گلیکسی زی فلپ جی ورژن ایک لائن ایونٹ پیش کیے جانے امک",
    "سلمان خان دوستوں دوستی لے ڈوبی ممبئی ویب ڈیسک بالی وڈ پنڈتوں کہنا بظاہر نظر پروڈکشن میدان اترنے سلمان خان پہلی بار جھٹکا لگنے اپنے دوستوں بچوں عطیہ شیٹی سورج پنچولی ان فلم ہیرو ریلیز ہو چکی اس رپورٹ اچھی نہیں تفصیلات سلمان خان فلم ہیرو اداکارہ کردار ",
    "ئی فون متعارف کرانے تاریخ سامنے گئی رواں سال شروع ایسی اطلاعات سامنے ئی تھیں ئی فون ایٹ اس بار معمول ستمبر سامنے نہیں سکے بلکہ اسے پیش کیا جائے گاتاہم ایپل اس نئے مکمل ری ڈیزائن اسمارٹ فون تاریخ رونمائی سامنے نہیں ئی مگر پتا چل گیا ستمبر عشرے متعارف ",
    "پاکستان ویسٹ انڈیز قسمت بدلنے خواہاں برج ٹان پاکستان عالمی چیمپیئن ویسٹ انڈیز درمیان چار ٹی میچوں سیریز اج اغاز ہونے جا جہاں میزبان ٹیم متحدہ عرب امارات گرین شرٹس بدترین کارکردگی بھلا بہترین کھیل پیش کرنے کوشش کرے گیگزشتہ سال ستمبر ان تین میچوں بے دا",
    "ایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم کراچیایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم ہوگیا کوارٹر فائنل شکست بعد محمد سجاد ابو صائم ایونٹ ہوگئے کراچی جاری ایونٹ کوارٹر فائنل ایران عامر سرکوش مقابلہ پاکستان محمد سجاد ایرانی کیوسٹ عمدہ کارکرد",
    "یو ایس اوپن ٹینس نوواک جوکووچ تیسرے رانڈ پہنچ گئے نیویارک اگست یو ایس اوپن ٹاپ سیڈ دفاعی چیمپئن نوواک جوکووچ باسانی تیسرے رانڈ پہنچ گئے اکہترویں فتح پیٹ سمپراس ہم پلہ بن گئے سرینا ولیمز میڈیسن کیز اپنے میچز سرخرو رہیں نیویارک ٹینس ایرینا جاری سیزن کا",
    "پی ٹی اے زونگ جی اشتہارات واپس لینے ہدایت پاکستان ٹیلی کمیونیکشن اتھارٹی پی ٹی اے چائنا موبائل پاکستان سی پی ایم یعنی زونگ جی ٹیکنالوجی اشتہارات تشہیر روکنے حکم ہےپی ٹی اے جانب جاری بیان زونگ جانب میڈیا حالیہ جی اشتہارات عوام گاہ کیا جاتا کمپنی فائیو",
    "ناقدین نظر بہترین فلمیں ناقدین نظر بہترین فلمیں امریکن گریفیٹی سائیکو سم لائیک اٹ ہاٹ منتھس ویکس اینڈ ڈیز گون ود ونڈ مائی لیفٹ فٹ ہوپ ڈریمز پینز لیبیرنتھ مون لائٹ سنگنگ ان رین تھری کلرز ریڈ بوائے ہڈ کاسا بلانکا گاڈ فادر سیٹزن کین ناقدین نظر بہترین فل",
    "پاکستانی اسکواڈ کرائسٹ چرچ کوئنز ٹان روانہ ہو لاہور دسمبر نیوزی لینڈ موجود قومی کرکٹ اسکواڈ مینیجڈ ئسولیشن چھوڑنے اجازت مل گئی قومی اسکواڈ شامل اکاون ارکان کرائسٹ چرچ مینیجڈ ئسولیشن چھوڑ دو ارکان اپنی ئسولیشن مدت پوری کریں نیوزی لینڈ موجود قومی اسکواڈ",
    "سرچ ہونے ڈیوائسز پسند ہو نہ ہو مگر ایپل ئی فون سکس ایس گوگل مقبول ڈیوائس قرار دے ہےگوگل جانب ٹرینڈز ٹیکنالوجی فہرست ئی فون سکس ایس سرچ ہونے باعث سرفہرست رہاویسے حیرت انگیز نہیں ئی فون نئے ماڈلز لگ بھگ سال بہترین قرار پاتے لوگوں نئے ئی فون جاننے تجسس ",
    "پاکستان اسٹاک ایکسچینج پوائنٹس اضافہ کراچی اپریل کاروباری ہفتے تیسرے روز پاکستان اسٹاک ایکسچینج مثبت رجحان انڈیکس پوائنٹس بہتری بدھ پاکستان اسٹاک ایکسچینج بہتری رجحان ایس ای انڈیکس پوائنٹس اضافے بعد ہزار پوائنٹس سطح پہنچ گیا کاروبار دوران کروڑ لاکھ ہ",
     ]
    # Execute all queries
    for idx, query in enumerate(queries, 1):
        print("\n\n" + "="*70)
        print(f"EXECUTING QUERY {idx} OF {len(queries)}")
        print("="*70)
        results = recommender.get_recommendations_comparison(
            query=query,
            n_results=50
        )
        recommender.compare_and_visualize(results, df=df, query_number=idx)
    # Save the final document
    recommender.save_document()
    print("\n" + "="*70)
    print("✓ COMPARISON SYSTEM DEMO COMPLETED!")
    print("✓ Full embeddings: 768 dimensions")
    print("✓ PCA embeddings: 64 dimensions")
    print(f"✓ All {len(queries)} queries executed")
    print(f"✓ Comprehensive report saved to: {recommender.output_doc_path}")
    print("✓ Visualizations saved as PNG files")
    print("="*70)

URDU NEWS RECOMMENDATION COMPARISON SYSTEM
Full Embeddings (768D) vs PCA Embeddings (64D)
OUTPUTTING TO WORD DOCUMENT

Loading dataset for content display...
✓ Dataset loaded: 111853 articles

INITIALIZING COMPARISON SYSTEM
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Loading PCA model from: chroma_db_mean_content_PCA64_claude/pca_model.pkl
✓ PCA model loaded (64 dimensions)
Connecting to Full Embeddings ChromaDB at: chroma_db_mean_content_full_claude
✓ Connected to full embeddings collection: urdu_news_embeddings_full_mean_claude
✓ Total articles in full database: 111853
Connecting to PCA Embeddings ChromaDB at: chroma_db_mean_content_PCA64_claude
✓ Connected to PCA embeddings collection: urdu_news_embeddings_pca64_mean_claude
✓ Total articles in PCA database: 111853

SYSTEM STATISTICS
Model: urduhack/roberta-urdu-small
Device: cuda
Pooling Method: MEAN POOLING

Full Embeddings (768D):
 total_articles: 111853
 collection_name: urdu_news_embeddings_full_mean_claude
 em

# **UMAP DR Claude**

In [6]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION WITH UMAP DIMENSIONALITY REDUCTION
# Creates two ChromaDB collections: full embeddings (768D) and UMAP reduced (64D)
# Using MEAN POOLING - chroma_db_mean_content_full & chroma_db_mean_content_UMAP64
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time
import umap
import pickle

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in two ChromaDB collections:
    1. Full embeddings (768 dimensions)
    2. UMAP-reduced embeddings (64 dimensions)

    Designed for large datasets (10,000+ records) with semantic search on content.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "./chroma_db_mean_content_full_claude",
                 chroma_db_path_umap: str = "./chroma_db_mean_content_UMAP64_claude",
                 umap_dimensions: int = 64):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path_full: Path to store ChromaDB with full embeddings
            chroma_db_path_umap: Path to store ChromaDB with UMAP-reduced embeddings
            umap_dimensions: Number of dimensions for UMAP reduction (default: 64)
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_umap = Path(chroma_db_path_umap)
        self.umap_dimensions = umap_dimensions
        self.umap_model = None

        # Create directories
        self.chroma_db_path_full.mkdir(exist_ok=True)
        self.chroma_db_path_umap.mkdir(exist_ok=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Initialize ChromaDB clients for both collections
        print(f"Initializing ChromaDB for full embeddings at: {self.chroma_db_path_full}")
        self.client_full = chromadb.PersistentClient(
            path=str(self.chroma_db_path_full)
        )

        print(f"Initializing ChromaDB for UMAP embeddings at: {self.chroma_db_path_umap}")
        self.client_umap = chromadb.PersistentClient(
            path=str(self.chroma_db_path_umap)
        )

        # Create or get collections for storing embeddings
        self.collection_full = self.client_full.get_or_create_collection(
            name="urdu_news_embeddings_full_mean_claude",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )

        self.collection_umap = self.client_umap.get_or_create_collection(
            name="urdu_news_embeddings_umap64_mean_UMAP_claude",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )

    def mean_pooling(self, model_output, attention_mask):
        """
        Apply mean pooling to model output to get sentence embeddings.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            Mean pooled embeddings (batch_size, embedding_dim)
        """
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512,
                                    chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for a single text using mean pooling.

        Args:
            text: Input Urdu text (content column)
            max_length: Maximum tokens per chunk (default: 512)
            chunk_overlap: Overlap between chunks to maintain context (default: 50)

        Returns:
            Embedding vector as numpy array (768,)
        """
        tokens = self.tokenizer.encode(text, add_special_tokens=True)

        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long articles: Split into overlapping chunks
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def fit_umap(self, embeddings_array: np.ndarray) -> None:
        """
        Fit UMAP model on the full embeddings.

        Args:
            embeddings_array: Array of shape (n_samples, embedding_dim)
        """
        print(f"\n{'='*70}")
        print(f"FITTING UMAP MODEL ({self.umap_dimensions} dimensions)")
        print(f"{'='*70}")
        print(f"Input shape: {embeddings_array.shape}")
        print(f"Note: UMAP fitting may take several minutes for large datasets...")

        # Initialize UMAP with parameters optimized for semantic embeddings
        self.umap_model = umap.UMAP(
            n_components=self.umap_dimensions,
            n_neighbors=15,  # Balance between local and global structure
            min_dist=0.1,    # Minimum distance between points
            metric='cosine', # Use cosine distance for semantic embeddings
            random_state=42,
            verbose=True     # Show progress
        )

        # Fit UMAP model
        fit_start = time.time()
        self.umap_model.fit(embeddings_array)
        fit_time = time.time() - fit_start

        print(f"✓ UMAP fitted successfully!")
        print(f"Fitting time: {fit_time:.2f} seconds ({fit_time/60:.2f} minutes)")
        print(f"Original dimensions: {embeddings_array.shape[1]}")
        print(f"Reduced dimensions: {self.umap_dimensions}")
        print(f"Metric: cosine")
        print(f"n_neighbors: 15")
        print(f"min_dist: 0.1")

        # Save UMAP model
        umap_model_path = self.chroma_db_path_umap / "umap_model.pkl"
        with open(umap_model_path, 'wb') as f:
            pickle.dump(self.umap_model, f)
        print(f"✓ UMAP model saved to: {umap_model_path}")

    def apply_umap(self, embeddings_array: np.ndarray) -> np.ndarray:
        """
        Apply UMAP transformation to embeddings.

        Args:
            embeddings_array: Array of shape (n_samples, embedding_dim)

        Returns:
            Reduced embeddings of shape (n_samples, umap_dimensions)
        """
        if self.umap_model is None:
            raise ValueError("UMAP model not fitted. Call fit_umap first.")

        print(f"Applying UMAP transformation...")
        transform_start = time.time()
        reduced_embeddings = self.umap_model.transform(embeddings_array)
        transform_time = time.time() - transform_start
        print(f"✓ Transformation complete in {transform_time:.2f} seconds")

        return reduced_embeddings

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       content_column: str = "content",
                                       headline_column: str = "Headline",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles and store in two ChromaDB collections:
        1. Full embeddings (768D)
        2. UMAP-reduced embeddings (64D)

        Args:
            df: Dataframe containing articles
            content_column: Column containing article content
            headline_column: Column containing article headline
            category_column: Column containing article category
        """
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Content column: '{content_column}' (used for semantic search)")
        print(f"Headline column: '{headline_column}' (stored as metadata)")
        print(f"Category column: '{category_column}' (stored as metadata)")
        print(f"Pooling method: MEAN POOLING")
        print(f"Full embeddings DB: chroma_db_mean_content_full_claude")
        print(f"UMAP embeddings DB: chroma_db_mean_content_UMAP64_claude")
        print(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data
        ids = []
        embeddings_full = []
        metadatas = []
        documents = []

        # Step 1: Generate full embeddings
        print("STEP 1: Generating full embeddings (768D)...")
        print("="*70)

        for idx, row in df.iterrows():
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed
                eta = (total_articles - idx - 1) / articles_per_sec
                print(f"Processed {idx + 1}/{total_articles} articles "
                      f"({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)")

            content_text = str(row[content_column])

            if len(content_text.strip()) == 0:
                print(f"Warning: Skipping article {idx} - empty content")
                continue

            try:
                # Generate full embedding
                embedding = self.generate_embedding_for_text(content_text)

                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings_full.append(embedding)
                documents.append(content_text[:500])

                metadata = {
                    "article_index": idx,
                    "headline": str(row.get(headline_column, "Unknown")),
                    "category": str(row.get(category_column, "Unknown")),
                    "content_length": len(content_text),
                    "pooling_method": "mean_pooling"
                }
                metadatas.append(metadata)

            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        # Convert to numpy array for UMAP
        embeddings_full_array = np.array(embeddings_full)

        # Step 2: Fit and apply UMAP
        self.fit_umap(embeddings_full_array)

        print(f"\n{'='*70}")
        print("STEP 2: Applying UMAP transformation...")
        print("="*70)

        embeddings_umap = self.apply_umap(embeddings_full_array)
        print(f"✓ UMAP transformation complete!")
        print(f"Reduced embeddings shape: {embeddings_umap.shape}")

        # Step 3: Store full embeddings in ChromaDB
        print(f"\n{'='*70}")
        print(f"STEP 3: STORING FULL EMBEDDINGS (768D) IN CHROMADB...")
        print(f"{'='*70}")

        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            # Add metadata indicating this is full embedding
            batch_metadatas = [
                {**meta, "embedding_type": "full", "dimensions": 768}
                for meta in metadatas[batch_idx:batch_end]
            ]

            self.collection_full.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings_full[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )

        print(f"✓ Full embeddings stored successfully!")

        # Step 4: Store UMAP embeddings in ChromaDB
        print(f"\n{'='*70}")
        print(f"STEP 4: STORING UMAP EMBEDDINGS (64D) IN CHROMADB...")
        print(f"{'='*70}")

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            # Add metadata indicating this is UMAP embedding
            batch_metadatas = [
                {**meta, "embedding_type": "umap", "dimensions": self.umap_dimensions}
                for meta in metadatas[batch_idx:batch_end]
            ]

            self.collection_umap.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings_umap[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )

        print(f"✓ UMAP embeddings stored successfully!")

        total_time = time.time() - start_time
        print(f"\n{'='*70}")
        print(f"✓ EMBEDDING GENERATION COMPLETE!")
        print(f"{'='*70}")
        print(f"Total embeddings stored: {len(ids)}")
        print(f"Full embeddings (768D): {len(ids)} in chroma_db_mean_content_full_claude")
        print(f"UMAP embeddings (64D): {len(ids)} in chroma_db_mean_content_UMAP64_claude")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Average time per article: {total_time/len(ids):.4f} seconds")
        print(f"Processing speed: {len(ids)/total_time:.2f} articles/second")

    def search_similar_articles(self, query_text: str, n_results: int = 5,
                               use_umap: bool = False) -> dict:
        """
        Search for similar articles using query text.

        Args:
            query_text: Query text to find similar articles
            n_results: Number of similar articles to return
            use_umap: If True, search in UMAP collection; if False, search in full collection

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Apply UMAP if needed
        if use_umap:
            if self.umap_model is None:
                # Try to load UMAP model
                umap_model_path = self.chroma_db_path_umap / "umap_model.pkl"
                if umap_model_path.exists():
                    with open(umap_model_path, 'rb') as f:
                        self.umap_model = pickle.load(f)
                else:
                    raise ValueError("UMAP model not found. Generate embeddings first.")

            query_embedding = self.umap_model.transform(query_embedding.reshape(1, -1))[0]
            collection = self.collection_umap
        else:
            collection = self.collection_full

        # Search in ChromaDB
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about both ChromaDB collections.

        Returns:
            Dictionary with collection information
        """
        count_full = self.collection_full.count()
        count_umap = self.collection_umap.count()

        return {
            "full_embeddings": {
                "total_embeddings": count_full,
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768,
                "db_path": str(self.chroma_db_path_full)
            },
            "umap_embeddings": {
                "total_embeddings": count_umap,
                "collection_name": self.collection_umap.name,
                "embedding_dimension": self.umap_dimensions,
                "db_path": str(self.chroma_db_path_umap),
                "metric": "cosine",
                "n_neighbors": 15,
                "min_dist": 0.1
            },
            "pooling_method": "MEAN POOLING",
            "search_column": "content"
        }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS EMBEDDINGS WITH UMAP DIMENSIONALITY REDUCTION")
    print("Full Embeddings: chroma_db_mean_content_full_claude (768D)")
    print("UMAP Embeddings: chroma_db_mean_content_UMAP64_claude (64D)")
    print("="*70)
    print("\nLoading dataset...")

    df = pd.read_csv("final_cleaned_urdu_news.csv")

    print(f"\nDataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset info:")
    print(f"  - Total articles: {len(df)}")
    print(f"  - Unique categories: {df['Category'].nunique()}")
    print(f"  - Categories: {df['Category'].unique().tolist()}")

    # Show sample data
    print(f"\n{'='*70}")
    print("SAMPLE DATA PREVIEW")
    print(f"{'='*70}")
    sample = df.iloc[0]
    print(f"Headline: {sample['Headline'][:100]}...")
    print(f"Category: {sample['Category']}")
    print(f"Content preview: {sample['content'][:200]}...")
    print(f"Content length: {len(sample['content'])} characters")

    # Initialize embedding generator
    print(f"\n{'='*70}")
    print("INITIALIZING EMBEDDING GENERATOR")
    print(f"{'='*70}")

    embedder = UrduNewsEmbeddingGenerator(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path_full="./chroma_db_mean_content_full_claude",
        chroma_db_path_umap="./chroma_db_mean_content_UMAP64_claude",
        umap_dimensions=64
    )

    # Generate embeddings and store in both ChromaDB collections
    embedder.generate_embeddings_for_dataset(
        df=df,
        content_column="content",
        headline_column="Headline",
        category_column="Category"
    )

    # Display collection statistics
    print("\n" + "="*70)
    print("CHROMADB COLLECTION STATISTICS")
    print("="*70)
    stats = embedder.get_collection_stats()
    print("\nFULL EMBEDDINGS COLLECTION:")
    for key, value in stats["full_embeddings"].items():
        print(f"  {key}: {value}")
    print("\nUMAP EMBEDDINGS COLLECTION:")
    for key, value in stats["umap_embeddings"].items():
        print(f"  {key}: {value}")
    print(f"\nPooling method: {stats['pooling_method']}")
    print(f"Search column: {stats['search_column']}")

    # Test: Search in both collections
    print("\n" + "="*70)
    print("TESTING SEMANTIC SEARCH")
    print("="*70)

    query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"
    print(f"\nQuery: {query}\n")

    # Search in full embeddings
    print("="*70)
    print("RESULTS FROM FULL EMBEDDINGS (768D)")
    print("="*70)
    results_full = embedder.search_similar_articles(query_text=query, n_results=5, use_umap=False)

    for i, (doc_id, distance, metadata) in enumerate(zip(
        results_full['ids'][0],
        results_full['distances'][0],
        results_full['metadatas'][0]
    )):
        print(f"\nResult #{i+1}")
        print(f"  Similarity Score: {1 - distance:.4f}")
        print(f"  Headline: {metadata.get('headline', 'N/A')[:80]}...")
        print(f"  Category: {metadata.get('category', 'N/A')}")

    # Search in UMAP embeddings
    print("\n" + "="*70)
    print("RESULTS FROM UMAP EMBEDDINGS (64D)")
    print("="*70)
    results_umap = embedder.search_similar_articles(query_text=query, n_results=5, use_umap=True)

    for i, (doc_id, distance, metadata) in enumerate(zip(
        results_umap['ids'][0],
        results_umap['distances'][0],
        results_umap['metadatas'][0]
    )):
        print(f"\nResult #{i+1}")
        print(f"  Similarity Score: {1 - distance:.4f}")
        print(f"  Headline: {metadata.get('headline', 'N/A')[:80]}...")
        print(f"  Category: {metadata.get('category', 'N/A')}")

    print("\n" + "="*70)
    print("✓ PROCESSING COMPLETED SUCCESSFULLY!")
    print("✓ Two ChromaDB collections created:")
    print("  1. chroma_db_mean_content_full_claude (768D)")
    print("  2. chroma_db_mean_content_UMAP64_claude (64D)")
    print("="*70)

URDU NEWS EMBEDDINGS WITH UMAP DIMENSIONALITY REDUCTION
Full Embeddings: chroma_db_mean_content_full_claude (768D)
UMAP Embeddings: chroma_db_mean_content_UMAP64_claude (64D)

Loading dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 4
  - Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا...
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters

INITIALIZING EMBEDDING GENERATOR
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing ChromaDB for full embeddings at: chroma_db_mean_cont

Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Sat Jan 17 19:52:17 2026 Finished embedding
✓ UMAP fitted successfully!
Fitting time: 154.40 seconds (2.57 minutes)
Original dimensions: 768
Reduced dimensions: 64
Metric: cosine
n_neighbors: 15
min_dist: 0.1
Sat Jan 17 19:52:18 2026 Building hub-based search tree
Sat Jan 17 19:52:22 2026 Forward diversification reduced edges from 1677795 to 645825
Sat Jan 17 19:52:23 2026 Reverse diversification reduced edges from 645825 to 645825
Sat Jan 17 19:52:25 2026 Degree pruning reduced edges from 765680 to 764250
Sat Jan 17 19:52:25 2026 Resorting data and graph based on tree order
Sat Jan 17 19:52:25 2026 Building and compiling search function
✓ UMAP model saved to: chroma_db_mean_content_UMAP

Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs

Result #1
  Similarity Score: 1.0000
  Headline: انٹرنیشنل ایگزیبیشن اف پلاسٹک مشینری21اپریل سے شروع ہوگی...
  Category: Business & Economics

Result #2
  Similarity Score: 1.0000
  Headline: ٹڈاپ پیپر ورلڈ اور لیدر ورلڈ مڈل ایسٹ میں شرکت کرے گی...
  Category: Business & Economics

Result #3
  Similarity Score: 1.0000
  Headline: ایکسپو سینٹر لاہور میں پاک چائنہ زرعی الات کی نمائش...
  Category: Business & Economics

Result #4
  Similarity Score: 1.0000
  Headline: کیا شہزادہ محمد بن سلمان واقعی مانچسٹر یونائیڈ کلب خرید رہے ہیں...
  Category: Sports

Result #5
  Similarity Score: 1.0000
  Headline: چار روزہ ایگاٹیکس پاکستان نمائش کا اج سے اغاز ہوگا...
  Category: Business & Economics

✓ PROC

# **Recommender System using UMAP DR**

In [2]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM WITH FULL VS UMAP EMBEDDINGS COMPARISON
# Compares recommendations from 768D full embeddings and 64D UMAP-reduced embeddings
# OUTPUTS RESULTS TO WORD DOCUMENT
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime
import umap.umap_ as umap
import warnings

warnings.filterwarnings('ignore')

# Configure matplotlib for better visualization
rcParams['figure.figsize'] = (16, 10)
rcParams['font.size'] = 10
sns.set_style("whitegrid")

class UrduNewsRecommenderComparison:
    """
    Recommendation system that compares results from full embeddings (768D)
    and UMAP-reduced embeddings (64D).
    Outputs results to Word document.
    """

    def __init__(self,
                 model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "chroma_db_mean_content_full_claude",
                 chroma_db_path_umap: str = "chroma_db_mean_content_UMAP64_claude",
                 collection_name_full: str = "urdu_news_embeddings_full_mean_claude",
                 collection_name_umap: str = "urdu_news_embeddings_umap64_mean_UMAP_claude",
                 output_doc_path: str = "Urdu_News_Full_vs_UMAP64_Comparison_Report.docx"):
        """
        Initialize recommender with connections to both full and UMAP databases.
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_umap = Path(chroma_db_path_umap)
        self.umap_model = None
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Load UMAP model
        umap_model_path = self.chroma_db_path_umap / "umap_model.pkl"
        if umap_model_path.exists():
            print(f"Loading UMAP model from: {umap_model_path}")
            self.add_paragraph(f"Loading UMAP model from: {umap_model_path}")
            with open(umap_model_path, 'rb') as f:
                self.umap_model = pickle.load(f)
            print(f"✓ UMAP model loaded (64 dimensions)")
            self.add_paragraph(f"✓ UMAP model loaded (64 dimensions)")
        else:
            raise FileNotFoundError(f"UMAP model not found at {umap_model_path}")

        # Connect to full embeddings ChromaDB
        print(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.add_paragraph(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.client_full = chromadb.PersistentClient(path=str(self.chroma_db_path_full))

        try:
            self.collection_full = self.client_full.get_collection(name=collection_name_full)
            print(f"✓ Connected to full embeddings collection: {collection_name_full}")
            print(f"✓ Total articles in full database: {self.collection_full.count()}")
            self.add_paragraph(f"✓ Connected to full embeddings collection: {collection_name_full}")
            self.add_paragraph(f"✓ Total articles in full database: {self.collection_full.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_full}'")
            raise e

        # Connect to UMAP embeddings ChromaDB
        print(f"Connecting to UMAP Embeddings ChromaDB at: {self.chroma_db_path_umap}")
        self.add_paragraph(f"Connecting to UMAP Embeddings ChromaDB at: {self.chroma_db_path_umap}")
        self.client_umap = chromadb.PersistentClient(path=str(self.chroma_db_path_umap))

        try:
            self.collection_umap = self.client_umap.get_collection(name=collection_name_umap)
            print(f"✓ Connected to UMAP embeddings collection: {collection_name_umap}")
            print(f"✓ Total articles in UMAP database: {self.collection_umap.count()}")
            self.add_paragraph(f"✓ Connected to UMAP embeddings collection: {collection_name_umap}")
            self.add_paragraph(f"✓ Total articles in UMAP database: {self.collection_umap.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_umap}'")
            raise e

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System\nFull (768D) vs UMAP (64D) Comparison Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text='', style=None):
        """Add paragraph to document."""
        para = self.doc.add_paragraph(text, style)
        return para

    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'

        for row_idx, row_data in enumerate(data):
            if headers and row_idx == 0:
                continue
            if not headers:
                cells = table.rows[row_idx].cells
            else:
                cells = table.add_row().cells
            for col_idx, cell_data in enumerate(row_data):
                cells[col_idx].text = str(cell_data)

        return table

    def add_image(self, image_path, width_inches=6):
        """Add image to document."""
        try:
            self.doc.add_picture(image_path, width=Inches(width_inches))
            last_paragraph = self.doc.paragraphs[-1]
            last_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
        except Exception as e:
            self.add_paragraph(f"Error adding image: {str(e)}")

    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"Word document saved to: {self.output_doc_path}")

    def mean_pooling(self, model_output, attention_mask):
        """Apply MEAN POOLING to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50, apply_umap: bool = False) -> np.ndarray:
        """
        Generate embedding for query text using MEAN POOLING.
        Optionally applies UMAP dimensionality reduction.
        """
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # Process short queries
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embedding = embeddings.cpu().detach().numpy()[0]

            # Apply UMAP if requested
            if apply_umap and self.umap_model is not None:
                embedding = self.umap_model.transform(embedding.reshape(1, -1))[0]

            return embedding

        # For long queries: use chunking
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)

        # Apply UMAP if requested
        if apply_umap and self.umap_model is not None:
            final_embedding = self.umap_model.transform(final_embedding.reshape(1, -1))[0]

        return final_embedding

    def get_recommendations_comparison(self, query: str, n_results: int = 10,
                                      filter_category: str = None) -> dict:
        """
        Get recommendations from both full and UMAP embeddings for comparison.
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS - FULL vs UMAP COMPARISON")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Generate embeddings for both spaces
        print("→ Generating query embedding (768D - Full)...")
        query_embedding_full = self.generate_query_embedding(query, apply_umap=False)
        print(f"  ✓ Full embedding generated: shape {query_embedding_full.shape}")

        print("→ Generating query embedding (64D - UMAP)...")
        query_embedding_umap = self.generate_query_embedding(query, apply_umap=True)
        print(f"  ✓ UMAP embedding generated: shape {query_embedding_umap.shape}")

        # Search in full embeddings
        print("\n→ Searching in Full Embeddings (768D)...")
        results_full = self.collection_full.query(
            query_embeddings=[query_embedding_full.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results_full['ids'][0])} recommendations")

        # Search in UMAP embeddings
        print("→ Searching in UMAP Embeddings (64D)...")
        results_umap = self.collection_umap.query(
            query_embeddings=[query_embedding_umap.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results_umap['ids'][0])} recommendations\n")

        return {
            'full': results_full,
            'umap': results_umap,
            'query': query,
            'n_results': n_results
        }

    def compare_and_visualize(self, comparison_results: dict, df: pd.DataFrame = None, query_number: int = 1):
        """
        Compare and visualize recommendations from full vs UMAP embeddings.
        Adds results to Word document.
        """
        results_full = comparison_results['full']
        results_umap = comparison_results['umap']
        query = comparison_results['query']
        n_results = comparison_results.get('n_results', 10)

        if not results_full['ids'] or not results_umap['ids']:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return

        # Add query heading to document
        self.add_heading(f"Query {query_number}: Comparison Results", level=1)
        self.add_paragraph(f"Query: {query[:200]}...")
        self.add_paragraph(f"Top {n_results} results from each method")
        self.add_paragraph('')

        # Extract data
        full_ids = results_full['ids'][0]
        full_distances = results_full['distances'][0]
        full_metadatas = results_full['metadatas'][0]

        umap_ids = results_umap['ids'][0]
        umap_distances = results_umap['distances'][0]
        umap_metadatas = results_umap['metadatas'][0]

        # Convert distances to similarity scores
        full_similarities = [1 - d for d in full_distances]
        umap_similarities = [1 - d for d in umap_distances]

        # Track overlap between results
        full_set = set(full_ids)
        umap_set = set(umap_ids)
        overlap = full_set.intersection(umap_set)

        # Print and add to doc: comparison summary
        print(f"\n{'='*70}")
        print(f"RECOMMENDATION COMPARISON SUMMARY - Query {query_number}")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...\n")
        print(f"Total recommendations per method: {n_results}")
        print(f"Articles in common: {len(overlap)} ({len(overlap)/n_results*100:.1f}%)")
        print(f"Unique to Full (768D): {len(full_set - umap_set)}")
        print(f"Unique to UMAP (64D): {len(umap_set - full_set)}")

        self.add_heading("Comparison Summary", level=2)
        summary_data = [
            ["Total recommendations per method", n_results],
            ["Articles in common", f"{len(overlap)} ({len(overlap)/n_results*100:.1f}%)"],
            ["Unique to Full (768D)", len(full_set - umap_set)],
            ["Unique to UMAP (64D)", len(umap_set - full_set)]
        ]
        self.add_table(summary_data, headers=["Metric", "Value"])
        self.add_paragraph('')

        # Create side-by-side comparison table for document
        self.add_heading("Side-by-Side Comparison", level=2)
        comparison_table_data = []

        for i in range(n_results):
            rank = i + 1

            # Full embedding info
            if i < len(full_ids):
                full_headline = full_metadatas[i].get('headline', 'N/A')[:80]
                full_sim = f"{full_similarities[i]:.4f}"
            else:
                full_headline = "N/A"
                full_sim = "0.0000"

            # UMAP embedding info
            if i < len(umap_ids):
                umap_headline = umap_metadatas[i].get('headline', 'N/A')[:80]
                umap_sim = f"{umap_similarities[i]:.4f}"
            else:
                umap_headline = "N/A"
                umap_sim = "0.0000"

            comparison_table_data.append([rank, full_headline, full_sim, umap_headline, umap_sim])

        self.add_table(comparison_table_data,
                      headers=["Rank", "Full (768D) Headline", "Full Similarity",
                              "UMAP (64D) Headline", "UMAP Similarity"])
        self.add_paragraph('')

        # Create and add visualization
        image_path = self._create_visualizations(overlap, full_set, umap_set, query, n_results, query_number)

        self.add_heading("Overlap Visualization", level=2)
        if image_path:
            self.add_image(image_path, width_inches=5)
        self.add_paragraph('')

    def _create_visualizations(self, overlap, full_set, umap_set, query, n_results, query_number):
        """Create comparison visualizations and return image path."""
        fig = plt.figure(figsize=(8, 6))

        # Overlap Analysis
        ax = plt.subplot(1, 1, 1)

        overlap_count = len(overlap)
        full_only = len(full_set - umap_set)
        umap_only = len(umap_set - full_set)

        sizes = [overlap_count, full_only, umap_only]
        labels = [f'Both\n({overlap_count})', f'Full Only\n({full_only})', f'UMAP Only\n({umap_only})']
        colors_pie = ['#4CAF50', '#2E86AB', '#F18F01']
        explode = (0.1, 0, 0)

        ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 10})
        ax.set_title('Recommendation Overlap Full Vs UMAP',
                     fontsize=12, fontweight='bold', pad=15)

        # Overall title
        fig.suptitle(f'Query {query_number} - Recommendation Overlap\nTop {n_results} Results',
                    fontsize=14, fontweight='bold', y=0.995)

        plt.tight_layout(rect=[0, 0.03, 1, 0.985])

        image_path = f'recommendation_comparison_umap64_query_{query_number}.png'
        plt.savefig(image_path, dpi=300, bbox_inches='tight')
        print(f"\n✓ Visualization saved as '{image_path}'")
        plt.close()

        return image_path

    def get_statistics(self) -> dict:
        """Get statistics about both recommendation systems."""
        umap_params = {}
        if self.umap_model:
            umap_params = {
                "n_neighbors": getattr(self.umap_model, 'n_neighbors', 'N/A'),
                "min_dist": getattr(self.umap_model, 'min_dist', 'N/A'),
                "metric": getattr(self.umap_model, 'metric', 'N/A')
            }

        return {
            "model": self.model_name,
            "device": str(self.device),
            "full_embeddings": {
                "total_articles": self.collection_full.count(),
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768
            },
            "umap_embeddings": {
                "total_articles": self.collection_umap.count(),
                "collection_name": self.collection_umap.name,
                "embedding_dimension": 64,
                "umap_parameters": umap_params
            },
            "pooling_method": "MEAN POOLING"
        }

    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        stats = self.get_statistics()

        stats_data = [
            ["Model", stats['model']],
            ["Device", stats['device']],
            ["Pooling Method", stats['pooling_method']]
        ]
        self.add_table(stats_data, headers=["Parameter", "Value"])

        self.add_heading("Full Embeddings (768D)", level=2)
        full_stats = [[k, v] for k, v in stats['full_embeddings'].items()]
        self.add_table(full_stats, headers=["Metric", "Value"])

        self.add_heading("UMAP Embeddings (64D)", level=2)
        umap_stats = []
        for k, v in stats['umap_embeddings'].items():
            if k == "umap_parameters":
                for param, param_value in v.items():
                    umap_stats.append([f"UMAP {param}", param_value])
            else:
                umap_stats.append([k, v])
        self.add_table(umap_stats, headers=["Metric", "Value"])
        self.add_paragraph('')


# =============================================================================
# MAIN EXECUTION - COMPARISON SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION COMPARISON SYSTEM")
    print("Full Embeddings (768D) vs UMAP Embeddings (64D)")
    print("OUTPUTTING TO WORD DOCUMENT")
    print("="*70)

    # Load dataset
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender
    print("\n" + "="*70)
    print("INITIALIZING COMPARISON SYSTEM")
    print("="*70)

    recommender = UrduNewsRecommenderComparison(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path_full="chroma_db_mean_content_full_claude",
        chroma_db_path_umap="chroma_db_mean_content_UMAP64_claude",
        collection_name_full="urdu_news_embeddings_full_mean_claude",
        collection_name_umap="urdu_news_embeddings_umap64_mean_UMAP_claude",
        output_doc_path="Urdu_News_Full_vs_UMAP64_Comparison_Report.docx"
    )

    # Display and add system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    print(f"Model: {stats['model']}")
    print(f"Device: {stats['device']}")
    print(f"Pooling Method: {stats['pooling_method']}")
    print(f"\nFull Embeddings (768D):")
    for key, value in stats['full_embeddings'].items():
        print(f"  {key}: {value}")
    print(f"\nUMAP Embeddings (64D):")
    for key, value in stats['umap_embeddings'].items():
        if key == "umap_parameters":
            print(f"  {key}:")
            for param, param_value in value.items():
                print(f"    {param}: {param_value}")
        else:
            print(f"  {key}: {value}")

    recommender.add_statistics_to_doc()

    # Define all queries
    queries = [
    "ریاضی اس سوال جواب دے پہلی نظر ریاضی سان سوال لگتا مگر اس انٹرنیٹ متعدد افراد ذہنوں پریشان کرکے رکھ ہےکیا اپنی ریاضی صلاحیت پورا بھروسا ہاں اس سوال جواب جس فیس بک صارفین ذہنوں گھما رکھ جس وجہ اس دو ممکنہ جواب ہیںاسے گو تھمبل تیار کیا اس فیس بک لوگوں ",
    "پاکستان اسٹاک ایکسچینج ملا جلا رجحان ہنڈرڈ انڈیکس پوائنٹس کمی بند پاکستان اسٹاک ایکسچینج اج کاروبار اختتام ملا جلا رجحان دیکھا گیا پاکستان اسٹاک ایکسچینج اج کاروبار اغاز اتار چڑھا جاری ملا جلا رجحان دیکھنے ایا اج کاروبار دوران کمپنیوں کروڑ شیئرز سودے",
    "سام سنگ نئے فلیگ شپ فون تاریخ رونمائی سامنے گئی سام سنگ اپنے نئے فلیگ شپ فون گلیکسی نوٹ ئندہ ماہ متعارف کرانے باضابطہ اعلان کردیا ہےسام سنگ جانب اگست گلیکسی نوٹ متعارف کرایا جائے جس گلیکسی زی فولڈ گلیکسی زی فلپ جی ورژن ایک لائن ایونٹ پیش کیے جانے امک",
    "سلمان خان دوستوں دوستی لے ڈوبی ممبئی ویب ڈیسک بالی وڈ پنڈتوں کہنا بظاہر نظر پروڈکشن میدان اترنے سلمان خان پہلی بار جھٹکا لگنے اپنے دوستوں بچوں عطیہ شیٹی سورج پنچولی ان فلم ہیرو ریلیز ہو چکی اس رپورٹ اچھی نہیں تفصیلات سلمان خان فلم ہیرو اداکارہ کردار ",
    "ئی فون متعارف کرانے تاریخ سامنے گئی رواں سال شروع ایسی اطلاعات سامنے ئی تھیں ئی فون ایٹ اس بار معمول ستمبر سامنے نہیں سکے بلکہ اسے پیش کیا جائے گاتاہم ایپل اس نئے مکمل ری ڈیزائن اسمارٹ فون تاریخ رونمائی سامنے نہیں ئی مگر پتا چل گیا ستمبر عشرے متعارف ",
    "پاکستان ویسٹ انڈیز قسمت بدلنے خواہاں برج ٹان پاکستان عالمی چیمپیئن ویسٹ انڈیز درمیان چار ٹی میچوں سیریز اج اغاز ہونے جا جہاں میزبان ٹیم متحدہ عرب امارات گرین شرٹس بدترین کارکردگی بھلا بہترین کھیل پیش کرنے کوشش کرے گیگزشتہ سال ستمبر ان تین میچوں بے دا",
    "ایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم کراچیایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم ہوگیا کوارٹر فائنل شکست بعد محمد سجاد ابو صائم ایونٹ ہوگئے کراچی جاری ایونٹ کوارٹر فائنل ایران عامر سرکوش مقابلہ پاکستان محمد سجاد ایرانی کیوسٹ عمدہ کارکرد",
    "یو ایس اوپن ٹینس نوواک جوکووچ تیسرے رانڈ پہنچ گئے نیویارک اگست یو ایس اوپن ٹاپ سیڈ دفاعی چیمپئن نوواک جوکووچ باسانی تیسرے رانڈ پہنچ گئے اکہترویں فتح پیٹ سمپراس ہم پلہ بن گئے سرینا ولیمز میڈیسن کیز اپنے میچز سرخرو رہیں نیویارک ٹینس ایرینا جاری سیزن کا",
    "پی ٹی اے زونگ جی اشتہارات واپس لینے ہدایت پاکستان ٹیلی کمیونیکشن اتھارٹی پی ٹی اے چائنا موبائل پاکستان سی پی ایم یعنی زونگ جی ٹیکنالوجی اشتہارات تشہیر روکنے حکم ہےپی ٹی اے جانب جاری بیان زونگ جانب میڈیا حالیہ جی اشتہارات عوام گاہ کیا جاتا کمپنی فائیو",
    "ناقدین نظر بہترین فلمیں ناقدین نظر بہترین فلمیں امریکن گریفیٹی سائیکو سم لائیک اٹ ہاٹ منتھس ویکس اینڈ ڈیز گون ود ونڈ مائی لیفٹ فٹ ہوپ ڈریمز پینز لیبیرنتھ مون لائٹ سنگنگ ان رین تھری کلرز ریڈ بوائے ہڈ کاسا بلانکا گاڈ فادر سیٹزن کین ناقدین نظر بہترین فل",
    "پاکستانی اسکواڈ کرائسٹ چرچ کوئنز ٹان روانہ ہو لاہور دسمبر نیوزی لینڈ موجود قومی کرکٹ اسکواڈ مینیجڈ ئسولیشن چھوڑنے اجازت مل گئی قومی اسکواڈ شامل اکاون ارکان کرائسٹ چرچ مینیجڈ ئسولیشن چھوڑ دو ارکان اپنی ئسولیشن مدت پوری کریں نیوزی لینڈ موجود قومی اسکوا",
    "سرچ ہونے ڈیوائسز پسند ہو نہ ہو مگر ایپل ئی فون سکس ایس گوگل مقبول ڈیوائس قرار دے ہےگوگل جانب ٹرینڈز ٹیکنالوجی فہرست ئی فون سکس ایس سرچ ہونے باعث سرفہرست رہاویسے حیرت انگیز نہیں ئی فون نئے ماڈلز لگ بھگ سال بہترین قرار پاتے لوگوں نئے ئی فون جاننے تجسس ",
    "پاکستان اسٹاک ایکسچینج پوائنٹس اضافہ کراچی اپریل کاروباری ہفتے تیسرے روز پاکستان اسٹاک ایکسچینج مثبت رجحان انڈیکس پوائنٹس بہتری بدھ پاکستان اسٹاک ایکسچینج بہتری رجحان ایس ای انڈیکس پوائنٹس اضافے بعد ہزار پوائنٹس سطح پہنچ گیا کاروبار دوران کروڑ لاکھ ہ",
     ]

    # Execute all queries
    for idx, query in enumerate(queries, 1):
        print("\n\n" + "="*70)
        print(f"EXECUTING QUERY {idx} OF {len(queries)}")
        print("="*70)

        results = recommender.get_recommendations_comparison(
            query=query,
            n_results=50
        )

        recommender.compare_and_visualize(results, df=df, query_number=idx)

    # Save the final document
    recommender.save_document()

    print("\n" + "="*70)
    print("✓ COMPARISON SYSTEM DEMO COMPLETED!")
    print("✓ Full embeddings: 768 dimensions")
    print("✓ UMAP embeddings: 64 dimensions")
    print(f"✓ All {len(queries)} queries executed")
    print(f"✓ Comprehensive report saved to: {recommender.output_doc_path}")
    print("✓ Visualizations saved as PNG files")
    print("="*70)

URDU NEWS RECOMMENDATION COMPARISON SYSTEM
Full Embeddings (768D) vs UMAP Embeddings (64D)
OUTPUTTING TO WORD DOCUMENT

Loading dataset for content display...
✓ Dataset loaded: 111853 articles

INITIALIZING COMPARISON SYSTEM
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Loading UMAP model from: chroma_db_mean_content_UMAP64_claude/umap_model.pkl
Fri Jan 30 19:21:56 2026 Building and compiling search function
✓ UMAP model loaded (64 dimensions)
Connecting to Full Embeddings ChromaDB at: chroma_db_mean_content_full_claude
✓ Connected to full embeddings collection: urdu_news_embeddings_full_mean_claude
✓ Total articles in full database: 111853
Connecting to UMAP Embeddings ChromaDB at: chroma_db_mean_content_UMAP64_claude
✓ Connected to UMAP embeddings collection: urdu_news_embeddings_umap64_mean_UMAP_claude
✓ Total articles in UMAP database: 111853

SYSTEM STATISTICS
Model: urduhack/roberta-urdu-small
Device: cuda
Pooling Method: MEAN POOLING

Full Embeddings (768D):
  to

Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
  ✓ UMAP embedding generated: shape (64,)

→ Searching in Full Embeddings (768D)...
  ✓ Found 50 recommendations
→ Searching in UMAP Embeddings (64D)...
  ✓ Found 50 recommendations


RECOMMENDATION COMPARISON SUMMARY - Query 1
Query: ریاضی اس سوال جواب دے پہلی نظر ریاضی سان سوال لگتا مگر اس انٹرنیٹ متعدد افراد ذہنوں پریشان کرکے رکھ ...

Total recommendations per method: 50
Articles in common: 23 (46.0%)
Unique to Full (768D): 27
Unique to UMAP (64D): 27

✓ Visualization saved as 'recommendation_comparison_umap64_query_1.png'


EXECUTING QUERY 2 OF 13

GENERATING RECOMMENDATIONS - FULL vs UMAP COMPARISON
Query: پاکستان اسٹاک ایکسچینج ملا جلا رجحان ہنڈرڈ انڈیکس پوائنٹس کمی بند پاکستان اسٹاک ای

Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
  ✓ UMAP embedding generated: shape (64,)

→ Searching in Full Embeddings (768D)...
  ✓ Found 50 recommendations
→ Searching in UMAP Embeddings (64D)...
  ✓ Found 50 recommendations


RECOMMENDATION COMPARISON SUMMARY - Query 2
Query: پاکستان اسٹاک ایکسچینج ملا جلا رجحان ہنڈرڈ انڈیکس پوائنٹس کمی بند پاکستان اسٹاک ایکسچینج اج کاروبار ...

Total recommendations per method: 50
Articles in common: 11 (22.0%)
Unique to Full (768D): 39
Unique to UMAP (64D): 39

✓ Visualization saved as 'recommendation_comparison_umap64_query_2.png'


EXECUTING QUERY 3 OF 13

GENERATING RECOMMENDATIONS - FULL vs UMAP COMPARISON
Query: سام سنگ نئے فلیگ شپ فون تاریخ رونمائی سامنے گئی سام سنگ اپنے نئے فلیگ شپ فون گلیکس

...

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
  ✓ UMAP embedding generated: shape (64,)

→ Searching in Full Embeddings (768D)...
  ✓ Found 50 recommendations
→ Searching in UMAP Embeddings (64D)...
  ✓ Found 50 recommendations


RECOMMENDATION COMPARISON SUMMARY - Query 11
Query: پاکستانی اسکواڈ کرائسٹ چرچ کوئنز ٹان روانہ ہو لاہور دسمبر نیوزی لینڈ موجود قومی کرکٹ اسکواڈ مینیجڈ ئ...

Total recommendations per method: 50
Articles in common: 8 (16.0%)
Unique to Full (768D): 42
Unique to UMAP (64D): 42

✓ Visualization saved as 'recommendation_comparison_umap64_query_11.png'


EXECUTING QUERY 12 OF 13

GENERATING RECOMMENDATIONS - FULL vs UMAP COMPARISON
Query: سرچ ہونے ڈیوائسز پسند ہو نہ ہو مگر ایپل ئی فون سکس ایس گوگل مقبول ڈیوائس قرار دے

Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
  ✓ UMAP embedding generated: shape (64,)

→ Searching in Full Embeddings (768D)...
  ✓ Found 50 recommendations
→ Searching in UMAP Embeddings (64D)...
  ✓ Found 50 recommendations


RECOMMENDATION COMPARISON SUMMARY - Query 12
Query: سرچ ہونے ڈیوائسز پسند ہو نہ ہو مگر ایپل ئی فون سکس ایس گوگل مقبول ڈیوائس قرار دے ہےگوگل جانب ٹرینڈز ...

Total recommendations per method: 50
Articles in common: 25 (50.0%)
Unique to Full (768D): 25
Unique to UMAP (64D): 25

✓ Visualization saved as 'recommendation_comparison_umap64_query_12.png'


EXECUTING QUERY 13 OF 13

GENERATING RECOMMENDATIONS - FULL vs UMAP COMPARISON
Query: پاکستان اسٹاک ایکسچینج پوائنٹس اضافہ کراچی اپریل کاروباری ہفتے تیسرے روز پاکستا

Epochs completed:   0%|            0/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs
  ✓ UMAP embedding generated: shape (64,)

→ Searching in Full Embeddings (768D)...
  ✓ Found 50 recommendations
→ Searching in UMAP Embeddings (64D)...
  ✓ Found 50 recommendations


RECOMMENDATION COMPARISON SUMMARY - Query 13
Query: پاکستان اسٹاک ایکسچینج پوائنٹس اضافہ کراچی اپریل کاروباری ہفتے تیسرے روز پاکستان اسٹاک ایکسچینج مثبت...

Total recommendations per method: 50
Articles in common: 19 (38.0%)
Unique to Full (768D): 31
Unique to UMAP (64D): 31

✓ Visualization saved as 'recommendation_comparison_umap64_query_13.png'
Word document saved to: Urdu_News_Full_vs_UMAP64_Comparison_Report.docx

✓ COMPARISON SYSTEM DEMO COMPLETED!
✓ Full embeddings: 768 dimensions
✓ UMAP embeddings: 64 di

# **LDA DR**

In [9]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION WITH LDA DIMENSIONALITY REDUCTION
# Creates two ChromaDB collections: full embeddings (768D) and LDA reduced (64D)
# Using MEAN POOLING - chroma_db_mean_content_full & chroma_db_mean_content_LDA64
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pickle

class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in two ChromaDB collections:
    1. Full embeddings (768 dimensions)
    2. LDA-reduced embeddings (64 dimensions)

    Designed for large datasets (10,000+ records) with semantic search on content.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "./chroma_db_mean_content_full_claude",
                 chroma_db_path_lda: str = "./chroma_db_mean_content_LDA64_claude",
                 lda_dimensions: int = 64):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path_full: Path to store ChromaDB with full embeddings
            chroma_db_path_lda: Path to store ChromaDB with LDA-reduced embeddings
            lda_dimensions: Number of dimensions for LDA reduction (default: 64)
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_lda = Path(chroma_db_path_lda)
        self.lda_dimensions = lda_dimensions
        self.lda_model = None

        # Create directories
        self.chroma_db_path_full.mkdir(exist_ok=True)
        self.chroma_db_path_lda.mkdir(exist_ok=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Initialize ChromaDB clients for both collections
        print(f"Initializing ChromaDB for full embeddings at: {self.chroma_db_path_full}")
        self.client_full = chromadb.PersistentClient(
            path=str(self.chroma_db_path_full)
        )

        print(f"Initializing ChromaDB for LDA embeddings at: {self.chroma_db_path_lda}")
        self.client_lda = chromadb.PersistentClient(
            path=str(self.chroma_db_path_lda)
        )

        # Create or get collections for storing embeddings
        self.collection_full = self.client_full.get_or_create_collection(
            name="urdu_news_embeddings_full_mean_claude",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )

        self.collection_lda = self.client_lda.get_or_create_collection(
            name="urdu_news_embeddings_lda64_mean_claude",
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )

    def mean_pooling(self, model_output, attention_mask):
        """
        Apply mean pooling to model output to get sentence embeddings.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            Mean pooled embeddings (batch_size, embedding_dim)
        """
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512,
                                    chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for a single text using mean pooling.

        Args:
            text: Input Urdu text (content column)
            max_length: Maximum tokens per chunk (default: 512)
            chunk_overlap: Overlap between chunks to maintain context (default: 50)

        Returns:
            Embedding vector as numpy array (768,)
        """
        tokens = self.tokenizer.encode(text, add_special_tokens=True)

        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embeddings = embeddings.cpu().detach().numpy()
            return embeddings[0]

        # For long articles: Split into overlapping chunks
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def fit_lda(self, embeddings_array: np.ndarray, labels: np.ndarray) -> None:
        """
        Fit LDA model on the full embeddings using category labels.

        Args:
            embeddings_array: Array of shape (n_samples, embedding_dim)
            labels: Array of category labels for supervised dimensionality reduction
        """
        print(f"\n{'='*70}")
        print(f"FITTING LDA MODEL ({self.lda_dimensions} dimensions)")
        print(f"{'='*70}")
        print(f"Input shape: {embeddings_array.shape}")
        print(f"Number of classes: {len(np.unique(labels))}")

        # LDA requires n_components < min(n_classes - 1, n_features)
        n_classes = len(np.unique(labels))
        max_components = min(n_classes - 1, embeddings_array.shape[1])

        if self.lda_dimensions >= max_components:
            print(f"Warning: Requested {self.lda_dimensions} components, but maximum is {max_components}")
            print(f"Setting n_components to {max_components}")
            self.lda_dimensions = max_components

        self.lda_model = LinearDiscriminantAnalysis(n_components=self.lda_dimensions)
        self.lda_model.fit(embeddings_array, labels)

        print(f"✓ LDA fitted successfully!")
        print(f"Original dimensions: {embeddings_array.shape[1]}")
        print(f"Reduced dimensions: {self.lda_dimensions}")

        # Calculate explained variance ratio if available
        if hasattr(self.lda_model, 'explained_variance_ratio_'):
            explained_variance = np.sum(self.lda_model.explained_variance_ratio_) * 100
            print(f"Explained variance: {explained_variance:.2f}%")

        # Save LDA model
        lda_model_path = self.chroma_db_path_lda / "lda_model.pkl"
        with open(lda_model_path, 'wb') as f:
            pickle.dump(self.lda_model, f)
        print(f"✓ LDA model saved to: {lda_model_path}")

    def apply_lda(self, embeddings_array: np.ndarray) -> np.ndarray:
        """
        Apply LDA transformation to embeddings.

        Args:
            embeddings_array: Array of shape (n_samples, embedding_dim)

        Returns:
            Reduced embeddings of shape (n_samples, lda_dimensions)
        """
        if self.lda_model is None:
            raise ValueError("LDA model not fitted. Call fit_lda first.")

        return self.lda_model.transform(embeddings_array)

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       content_column: str = "content",
                                       headline_column: str = "Headline",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles and store in two ChromaDB collections:
        1. Full embeddings (768D)
        2. LDA-reduced embeddings (64D)

        Args:
            df: Dataframe containing articles
            content_column: Column containing article content
            headline_column: Column containing article headline
            category_column: Column containing article category (used for LDA)
        """
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")
        print(f"Content column: '{content_column}' (used for semantic search)")
        print(f"Headline column: '{headline_column}' (stored as metadata)")
        print(f"Category column: '{category_column}' (used for LDA and stored as metadata)")
        print(f"Pooling method: MEAN POOLING")
        print(f"Dimensionality Reduction: LDA (supervised)")
        print(f"Full embeddings DB: chroma_db_mean_content_full")
        print(f"LDA embeddings DB: chroma_db_mean_content_LDA64")
        print(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data
        ids = []
        embeddings_full = []
        metadatas = []
        documents = []
        labels = []

        # Step 1: Generate full embeddings
        print("STEP 1: Generating full embeddings (768D)...")
        print("="*70)

        for idx, row in df.iterrows():
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed
                eta = (total_articles - idx - 1) / articles_per_sec
                print(f"Processed {idx + 1}/{total_articles} articles "
                      f"({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)")

            content_text = str(row[content_column])

            if len(content_text.strip()) == 0:
                print(f"Warning: Skipping article {idx} - empty content")
                continue

            try:
                # Generate full embedding
                embedding = self.generate_embedding_for_text(content_text)

                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings_full.append(embedding)
                documents.append(content_text[:500])
                labels.append(str(row[category_column]))

                metadata = {
                    "article_index": idx,
                    "headline": str(row.get(headline_column, "Unknown")),
                    "category": str(row.get(category_column, "Unknown")),
                    "content_length": len(content_text),
                    "pooling_method": "mean_pooling"
                }
                metadatas.append(metadata)

            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        # Convert to numpy arrays
        embeddings_full_array = np.array(embeddings_full)
        labels_array = np.array(labels)

        # Encode labels to numeric
        from sklearn.preprocessing import LabelEncoder
        label_encoder = LabelEncoder()
        labels_encoded = label_encoder.fit_transform(labels_array)

        # Save label encoder
        label_encoder_path = self.chroma_db_path_lda / "label_encoder.pkl"
        with open(label_encoder_path, 'wb') as f:
            pickle.dump(label_encoder, f)

        # Step 2: Fit and apply LDA
        self.fit_lda(embeddings_full_array, labels_encoded)

        print(f"\n{'='*70}")
        print("STEP 2: Applying LDA transformation...")
        print("="*70)

        embeddings_lda = self.apply_lda(embeddings_full_array)
        print(f"✓ LDA transformation complete!")
        print(f"Reduced embeddings shape: {embeddings_lda.shape}")

        # Step 3: Store full embeddings in ChromaDB
        print(f"\n{'='*70}")
        print(f"STEP 3: STORING FULL EMBEDDINGS (768D) IN CHROMADB...")
        print(f"{'='*70}")

        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            # Add metadata indicating this is full embedding
            batch_metadatas = [
                {**meta, "embedding_type": "full", "dimensions": 768}
                for meta in metadatas[batch_idx:batch_end]
            ]

            self.collection_full.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings_full[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )

        print(f"✓ Full embeddings stored successfully!")

        # Step 4: Store LDA embeddings in ChromaDB
        print(f"\n{'='*70}")
        print(f"STEP 4: STORING LDA EMBEDDINGS ({self.lda_dimensions}D) IN CHROMADB...")
        print(f"{'='*70}")

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            print(f"Storing batch {current_batch}/{total_batches} "
                  f"(items {batch_idx} to {batch_end})...")

            # Add metadata indicating this is LDA embedding
            batch_metadatas = [
                {**meta, "embedding_type": "lda", "dimensions": self.lda_dimensions}
                for meta in metadatas[batch_idx:batch_end]
            ]

            self.collection_lda.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings_lda[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )

        print(f"✓ LDA embeddings stored successfully!")

        total_time = time.time() - start_time
        print(f"\n{'='*70}")
        print(f"✓ EMBEDDING GENERATION COMPLETE!")
        print(f"{'='*70}")
        print(f"Total embeddings stored: {len(ids)}")
        print(f"Full embeddings (768D): {len(ids)} in chroma_db_mean_content_full")
        print(f"LDA embeddings ({self.lda_dimensions}D): {len(ids)} in chroma_db_mean_content_LDA64")
        print(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        print(f"Average time per article: {total_time/len(ids):.4f} seconds")
        print(f"Processing speed: {len(ids)/total_time:.2f} articles/second")

    def search_similar_articles(self, query_text: str, n_results: int = 5,
                               use_lda: bool = False) -> dict:
        """
        Search for similar articles using query text.

        Args:
            query_text: Query text to find similar articles
            n_results: Number of similar articles to return
            use_lda: If True, search in LDA collection; if False, search in full collection

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Apply LDA if needed
        if use_lda:
            if self.lda_model is None:
                # Try to load LDA model
                lda_model_path = self.chroma_db_path_lda / "lda_model.pkl"
                if lda_model_path.exists():
                    with open(lda_model_path, 'rb') as f:
                        self.lda_model = pickle.load(f)
                else:
                    raise ValueError("LDA model not found. Generate embeddings first.")

            query_embedding = self.lda_model.transform(query_embedding.reshape(1, -1))[0]
            collection = self.collection_lda
        else:
            collection = self.collection_full

        # Search in ChromaDB
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about both ChromaDB collections.

        Returns:
            Dictionary with collection information
        """
        count_full = self.collection_full.count()
        count_lda = self.collection_lda.count()

        explained_var = "N/A"
        if self.lda_model and hasattr(self.lda_model, 'explained_variance_ratio_'):
            explained_var = f"{np.sum(self.lda_model.explained_variance_ratio_) * 100:.2f}%"

        return {
            "full_embeddings": {
                "total_embeddings": count_full,
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768,
                "db_path": str(self.chroma_db_path_full)
            },
            "lda_embeddings": {
                "total_embeddings": count_lda,
                "collection_name": self.collection_lda.name,
                "embedding_dimension": self.lda_dimensions,
                "db_path": str(self.chroma_db_path_lda),
                "explained_variance": explained_var
            },
            "pooling_method": "MEAN POOLING",
            "dimensionality_reduction": "LDA (supervised)",
            "search_column": "content"
        }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS EMBEDDINGS WITH LDA DIMENSIONALITY REDUCTION")
    print("Full Embeddings: chroma_db_mean_content_full (768D)")
    print("LDA Embeddings: chroma_db_mean_content_LDA64 (64D)")
    print("="*70)
    print("\nLoading dataset...")

    df = pd.read_csv("final_cleaned_urdu_news.csv")

    print(f"\nDataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDataset info:")
    print(f"  - Total articles: {len(df)}")
    print(f"  - Unique categories: {df['Category'].nunique()}")
    print(f"  - Categories: {df['Category'].unique().tolist()}")

    # Show sample data
    print(f"\n{'='*70}")
    print("SAMPLE DATA PREVIEW")
    print(f"{'='*70}")
    sample = df.iloc[0]
    print(f"Headline: {sample['Headline'][:100]}...")
    print(f"Category: {sample['Category']}")
    print(f"Content preview: {sample['content'][:200]}...")
    print(f"Content length: {len(sample['content'])} characters")

    # Initialize embedding generator
    print(f"\n{'='*70}")
    print("INITIALIZING EMBEDDING GENERATOR")
    print(f"{'='*70}")

    embedder = UrduNewsEmbeddingGenerator(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path_full="./chroma_db_mean_content_full_claude",
        chroma_db_path_lda="./chroma_db_mean_content_LDA64_claude",
        lda_dimensions=64
    )

    # Generate embeddings and store in both ChromaDB collections
    embedder.generate_embeddings_for_dataset(
        df=df,
        content_column="content",
        headline_column="Headline",
        category_column="Category"
    )

    # Display collection statistics
    print("\n" + "="*70)
    print("CHROMADB COLLECTION STATISTICS")
    print("="*70)
    stats = embedder.get_collection_stats()
    print("\nFULL EMBEDDINGS COLLECTION:")
    for key, value in stats["full_embeddings"].items():
        print(f"  {key}: {value}")
    print("\nLDA EMBEDDINGS COLLECTION:")
    for key, value in stats["lda_embeddings"].items():
        print(f"  {key}: {value}")
    print(f"\nPooling method: {stats['pooling_method']}")
    print(f"Dimensionality reduction: {stats['dimensionality_reduction']}")
    print(f"Search column: {stats['search_column']}")

    # Test: Search in both collections
    print("\n" + "="*70)
    print("TESTING SEMANTIC SEARCH")
    print("="*70)

    query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"
    print(f"\nQuery: {query}\n")

    # Search in full embeddings
    print("="*70)
    print("RESULTS FROM FULL EMBEDDINGS (768D)")
    print("="*70)
    results_full = embedder.search_similar_articles(query_text=query, n_results=5, use_lda=False)

    for i, (doc_id, distance, metadata) in enumerate(zip(
        results_full['ids'][0],
        results_full['distances'][0],
        results_full['metadatas'][0]
    )):
        print(f"\nResult #{i+1}")
        print(f"  Similarity Score: {1 - distance:.4f}")
        print(f"  Headline: {metadata.get('headline', 'N/A')[:80]}...")
        print(f"  Category: {metadata.get('category', 'N/A')}")

    # Search in LDA embeddings
    print("\n" + "="*70)
    print(f"RESULTS FROM LDA EMBEDDINGS ({embedder.lda_dimensions}D)")
    print("="*70)
    results_lda = embedder.search_similar_articles(query_text=query, n_results=5, use_lda=True)

    for i, (doc_id, distance, metadata) in enumerate(zip(
        results_lda['ids'][0],
        results_lda['distances'][0],
        results_lda['metadatas'][0]
    )):
        print(f"\nResult #{i+1}")
        print(f"  Similarity Score: {1 - distance:.4f}")
        print(f"  Headline: {metadata.get('headline', 'N/A')[:80]}...")
        print(f"  Category: {metadata.get('category', 'N/A')}")

    print("\n" + "="*70)
    print("✓ PROCESSING COMPLETED SUCCESSFULLY!")
    print("✓ Two ChromaDB collections created:")
    print("  1. chroma_db_mean_content_full (768D)")
    print(f"  2. chroma_db_mean_content_LDA64 ({embedder.lda_dimensions}D)")
    print("="*70)

URDU NEWS EMBEDDINGS WITH LDA DIMENSIONALITY REDUCTION
Full Embeddings: chroma_db_mean_content_full (768D)
LDA Embeddings: chroma_db_mean_content_LDA64 (64D)

Loading dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 4
  - Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا...
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters

INITIALIZING EMBEDDING GENERATOR
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing ChromaDB for full embeddings at: chroma_db_mean_content_full_claude
I

# **LDA Recommender Code Content Col**

In [10]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM WITH FULL VS LDA EMBEDDINGS COMPARISON
# Compares recommendations from 768D full embeddings and 64D LDA-reduced embeddings
# OUTPUTS RESULTS TO WORD DOCUMENT
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import warnings

warnings.filterwarnings('ignore')

# Configure matplotlib for better visualization
rcParams['figure.figsize'] = (16, 10)
rcParams['font.size'] = 10
sns.set_style("whitegrid")

class UrduNewsRecommenderComparison:
    """
    Recommendation system that compares results from full embeddings (768D)
    and LDA-reduced embeddings (64D).
    Outputs results to Word document.
    """

    def __init__(self,
                 model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "./chroma_db_mean_content_full_claude",
                 chroma_db_path_lda: str = "./chroma_db_mean_content_LDA64_claude",
                 collection_name_full: str = "urdu_news_embeddings_full_mean_claude",
                 collection_name_lda: str = "urdu_news_embeddings_lda64_mean_claude",
                 output_doc_path: str = "Urdu_News_Full_vs_LDA_Comparison_Report.docx"):
        """
        Initialize recommender with connections to both full and LDA databases.
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_lda = Path(chroma_db_path_lda)
        self.lda_model = None
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Load LDA model
        lda_model_path = self.chroma_db_path_lda / "lda_model.pkl"
        if lda_model_path.exists():
            print(f"Loading LDA model from: {lda_model_path}")
            self.add_paragraph(f"Loading LDA model from: {lda_model_path}")
            with open(lda_model_path, 'rb') as f:
                self.lda_model = pickle.load(f)
            print(f"✓ LDA model loaded ({self.lda_model.n_components if hasattr(self.lda_model, 'n_components') else 'N/A'} dimensions)")
            self.add_paragraph(f"✓ LDA model loaded ({self.lda_model.n_components if hasattr(self.lda_model, 'n_components') else 'N/A'} dimensions)")
        else:
            raise FileNotFoundError(f"LDA model not found at {lda_model_path}")

        # Connect to full embeddings ChromaDB
        print(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.add_paragraph(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.client_full = chromadb.PersistentClient(path=str(self.chroma_db_path_full))

        try:
            self.collection_full = self.client_full.get_collection(name=collection_name_full)
            print(f"✓ Connected to full embeddings collection: {collection_name_full}")
            print(f"✓ Total articles in full database: {self.collection_full.count()}")
            self.add_paragraph(f"✓ Connected to full embeddings collection: {collection_name_full}")
            self.add_paragraph(f"✓ Total articles in full database: {self.collection_full.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_full}'")
            raise e

        # Connect to LDA embeddings ChromaDB
        print(f"Connecting to LDA Embeddings ChromaDB at: {self.chroma_db_path_lda}")
        self.add_paragraph(f"Connecting to LDA Embeddings ChromaDB at: {self.chroma_db_path_lda}")
        self.client_lda = chromadb.PersistentClient(path=str(self.chroma_db_path_lda))

        try:
            self.collection_lda = self.client_lda.get_collection(name=collection_name_lda)
            print(f"✓ Connected to LDA embeddings collection: {collection_name_lda}")
            print(f"✓ Total articles in LDA database: {self.collection_lda.count()}")
            self.add_paragraph(f"✓ Connected to LDA embeddings collection: {collection_name_lda}")
            self.add_paragraph(f"✓ Total articles in LDA database: {self.collection_lda.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_lda}'")
            raise e

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System\nFull (768D) vs LDA (64D) Comparison Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text='', style=None):
        """Add paragraph to document."""
        para = self.doc.add_paragraph(text, style)
        return para

    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
            
            # Add data rows
            for row_data in data:
                cells = table.add_row().cells
                for col_idx, cell_data in enumerate(row_data):
                    cells[col_idx].text = str(cell_data)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'
            
            for row_idx, row_data in enumerate(data):
                cells = table.rows[row_idx].cells
                for col_idx, cell_data in enumerate(row_data):
                    cells[col_idx].text = str(cell_data)

        return table

    def add_image(self, image_path, width_inches=6):
        """Add image to document."""
        try:
            self.doc.add_picture(image_path, width=Inches(width_inches))
            last_paragraph = self.doc.paragraphs[-1]
            last_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
            self.add_paragraph('')  # Add empty line after image
        except Exception as e:
            self.add_paragraph(f"Error adding image: {str(e)}")

    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"Word document saved to: {self.output_doc_path}")

    def mean_pooling(self, model_output, attention_mask):
        """Apply MEAN POOLING to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50, apply_lda: bool = False) -> np.ndarray:
        """
        Generate embedding for query text using MEAN POOLING.
        Optionally applies LDA dimensionality reduction.
        """
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # Process short queries
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embedding = embeddings.cpu().detach().numpy()[0]

            # Apply LDA if requested
            if apply_lda and self.lda_model is not None:
                embedding = self.lda_model.transform(embedding.reshape(1, -1))[0]

            return embedding

        # For long queries: use chunking
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)

        # Apply LDA if requested
        if apply_lda and self.lda_model is not None:
            final_embedding = self.lda_model.transform(final_embedding.reshape(1, -1))[0]

        return final_embedding

    def get_recommendations_comparison(self, query: str, n_results: int = 10,
                                      filter_category: str = None) -> dict:
        """
        Get recommendations from both full and LDA embeddings for comparison.
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS - FULL vs LDA COMPARISON")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Generate embeddings for both spaces
        print("→ Generating query embedding (768D - Full)...")
        query_embedding_full = self.generate_query_embedding(query, apply_lda=False)
        print(f"  ✓ Full embedding generated: shape {query_embedding_full.shape}")

        print("→ Generating query embedding (64D - LDA)...")
        query_embedding_lda = self.generate_query_embedding(query, apply_lda=True)
        print(f"  ✓ LDA embedding generated: shape {query_embedding_lda.shape}")

        # Search in full embeddings
        print("\n→ Searching in Full Embeddings (768D)...")
        results_full = self.collection_full.query(
            query_embeddings=[query_embedding_full.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results_full['ids'][0])} recommendations")

        # Search in LDA embeddings
        print("→ Searching in LDA Embeddings (64D)...")
        results_lda = self.collection_lda.query(
            query_embeddings=[query_embedding_lda.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results_lda['ids'][0])} recommendations\n")

        return {
            'full': results_full,
            'lda': results_lda,
            'query': query,
            'n_results': n_results
        }

    def compare_and_visualize(self, comparison_results: dict, df: pd.DataFrame = None, query_number: int = 1):
        """
        Compare and visualize recommendations from full vs LDA embeddings.
        Adds results to Word document.
        """
        results_full = comparison_results['full']
        results_lda = comparison_results['lda']
        query = comparison_results['query']
        n_results = comparison_results.get('n_results', 10)

        if not results_full['ids'] or not results_lda['ids']:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return

        # Add query heading to document
        self.add_heading(f"Query {query_number}: Comparison Results", level=1)
        self.add_paragraph(f"Query: {query[:200]}...")
        self.add_paragraph(f"Top {n_results} results from each method")
        self.add_paragraph('')

        # Extract data
        full_ids = results_full['ids'][0]
        full_distances = results_full['distances'][0]
        full_metadatas = results_full['metadatas'][0]

        lda_ids = results_lda['ids'][0]
        lda_distances = results_lda['distances'][0]
        lda_metadatas = results_lda['metadatas'][0]

        # Convert distances to similarity scores
        full_similarities = [1 - d for d in full_distances]
        lda_similarities = [1 - d for d in lda_distances]

        # Track overlap between results
        full_set = set(full_ids)
        lda_set = set(lda_ids)
        overlap = full_set.intersection(lda_set)

        # Print and add to doc: comparison summary
        print(f"\n{'='*70}")
        print(f"RECOMMENDATION COMPARISON SUMMARY - Query {query_number}")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...\n")
        print(f"Total recommendations per method: {n_results}")
        print(f"Articles in common: {len(overlap)} ({len(overlap)/n_results*100:.1f}%)")
        print(f"Unique to Full (768D): {len(full_set - lda_set)}")
        print(f"Unique to LDA (64D): {len(lda_set - full_set)}")

        self.add_heading("Comparison Summary", level=2)
        summary_data = [
            ["Total recommendations per method", n_results],
            ["Articles in common", f"{len(overlap)} ({len(overlap)/n_results*100:.1f}%)"],
            ["Unique to Full (768D)", len(full_set - lda_set)],
            ["Unique to LDA (64D)", len(lda_set - full_set)]
        ]
        self.add_table(summary_data, headers=["Metric", "Value"])
        self.add_paragraph('')

        # Create side-by-side comparison table for document
        self.add_heading("Side-by-Side Comparison", level=2)
        comparison_table_data = []

        for i in range(n_results):
            rank = i + 1

            # Full embedding info
            if i < len(full_ids):
                full_headline = full_metadatas[i].get('headline', 'N/A')[:80]
                full_sim = f"{full_similarities[i]:.4f}"
            else:
                full_headline = "N/A"
                full_sim = "0.0000"

            # LDA embedding info
            if i < len(lda_ids):
                lda_headline = lda_metadatas[i].get('headline', 'N/A')[:80]
                lda_sim = f"{lda_similarities[i]:.4f}"
            else:
                lda_headline = "N/A"
                lda_sim = "0.0000"

            comparison_table_data.append([rank, full_headline, full_sim, lda_headline, lda_sim])

        self.add_table(comparison_table_data,
                      headers=["Rank", "Full (768D) Headline", "Full Similarity",
                              "LDA (64D) Headline", "LDA Similarity"])
        self.add_paragraph('')

        # Create and add visualization
        image_path = self._create_visualizations(overlap, full_set, lda_set, query, n_results, query_number)

        self.add_heading("Overlap Visualization", level=2)
        if image_path:
            self.add_image(image_path, width_inches=5)
        self.add_paragraph('')

    def _create_visualizations(self, overlap, full_set, lda_set, query, n_results, query_number):
        """Create comparison visualizations and return image path."""
        fig = plt.figure(figsize=(8, 6))

        # Overlap Analysis
        ax = plt.subplot(1, 1, 1)

        overlap_count = len(overlap)
        full_only = len(full_set - lda_set)
        lda_only = len(lda_set - full_set)

        sizes = [overlap_count, full_only, lda_only]
        labels = [f'Both\n({overlap_count})', f'Full Only\n({full_only})', f'LDA Only\n({lda_only})']
        colors_pie = ['#4CAF50', '#2E86AB', '#F18F01']
        explode = (0.1, 0, 0)

        ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 10})
        ax.set_title('Recommendation Overlap: Full (768D) vs LDA (64D)',
                     fontsize=12, fontweight='bold', pad=15)

        # Overall title
        fig.suptitle(f'Query {query_number} - Recommendation Overlap\nTop {n_results} Results',
                    fontsize=14, fontweight='bold', y=0.995)

        plt.tight_layout(rect=[0, 0.03, 1, 0.985])

        image_path = f'recommendation_comparison_lda_query_{query_number}.png'
        plt.savefig(image_path, dpi=300, bbox_inches='tight')
        print(f"\n✓ Visualization saved as '{image_path}'")
        plt.close()

        return image_path

    def get_statistics(self) -> dict:
        """Get statistics about both recommendation systems."""
        lda_dimensions = 64  # Fixed to 64D based on your specification
        
        lda_params = {}
        if self.lda_model:
            lda_params = {
                "n_components": lda_dimensions,
                "solver": getattr(self.lda_model, 'solver', 'N/A'),
                "shrinkage": getattr(self.lda_model, 'shrinkage', 'N/A')
            }

        return {
            "model": self.model_name,
            "device": str(self.device),
            "full_embeddings": {
                "total_articles": self.collection_full.count(),
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768
            },
            "lda_embeddings": {
                "total_articles": self.collection_lda.count(),
                "collection_name": self.collection_lda.name,
                "embedding_dimension": lda_dimensions,
                "lda_parameters": lda_params
            },
            "pooling_method": "MEAN POOLING"
        }

    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        stats = self.get_statistics()

        stats_data = [
            ["Model", stats['model']],
            ["Device", stats['device']],
            ["Pooling Method", stats['pooling_method']]
        ]
        self.add_table(stats_data, headers=["Parameter", "Value"])

        self.add_heading("Full Embeddings (768D)", level=2)
        full_stats = [[k, v] for k, v in stats['full_embeddings'].items()]
        self.add_table(full_stats, headers=["Metric", "Value"])

        self.add_heading("LDA Embeddings (64D)", level=2)
        lda_stats = []
        for k, v in stats['lda_embeddings'].items():
            if k == "lda_parameters":
                for param, param_value in v.items():
                    lda_stats.append([f"LDA {param}", param_value])
            else:
                lda_stats.append([k, v])
        self.add_table(lda_stats, headers=["Metric", "Value"])
        self.add_paragraph('')


# =============================================================================
# MAIN EXECUTION - COMPARISON SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION COMPARISON SYSTEM")
    print("Full Embeddings (768D) vs LDA Embeddings (64D)")
    print("OUTPUTTING TO WORD DOCUMENT")
    print("="*70)

    # Load dataset
    print("\nLoading dataset for content display...")
    df = pd.read_csv("final_cleaned_urdu_news.csv")
    
    # Clean the data - handle NaN values in key columns
    print("Cleaning dataset...")
    df['Category'] = df['Category'].fillna('Unknown')
    df['Headline'] = df['Headline'].fillna('Unknown')
    df['content'] = df['content'].fillna('')
    
    print(f"✓ Dataset loaded: {len(df)} articles")

    # Initialize recommender
    print("\n" + "="*70)
    print("INITIALIZING COMPARISON SYSTEM")
    print("="*70)

    recommender = UrduNewsRecommenderComparison(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path_full="./chroma_db_mean_content_full_claude",
        chroma_db_path_lda="./chroma_db_mean_content_LDA64_claude",
        collection_name_full="urdu_news_embeddings_full_mean_claude",
        collection_name_lda="urdu_news_embeddings_lda64_mean_claude",
        output_doc_path="Urdu_News_Full_vs_LDA_Comparison_Report.docx"
    )

    # Add dataset information to document with NaN handling
    recommender.add_heading("Dataset Information", level=1)
    
    # Get unique categories, handle NaN values
    categories = df['Category'].unique().tolist()
    # Convert all items to string, handle NaN
    categories_str = [str(cat) if pd.notna(cat) else "Unknown" for cat in categories]
    
    dataset_info = [
        ["Total articles", len(df)],
        ["Dataset shape", f"{df.shape}"],
        ["Unique categories", df['Category'].nunique()],
        ["Categories", ", ".join(categories_str[:15]) + ("..." if len(categories_str) > 15 else "")]
    ]
    recommender.add_table(dataset_info, headers=["Metric", "Value"])

    # Display and add system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    print(f"Model: {stats['model']}")
    print(f"Device: {stats['device']}")
    print(f"Pooling Method: {stats['pooling_method']}")
    print(f"\nFull Embeddings (768D):")
    for key, value in stats['full_embeddings'].items():
        print(f"  {key}: {value}")
    print(f"\nLDA Embeddings (64D):")
    for key, value in stats['lda_embeddings'].items():
        if key == "lda_parameters":
            print(f"  {key}:")
            for param, param_value in value.items():
                print(f"    {param}: {param_value}")
        else:
            print(f"  {key}: {value}")

    recommender.add_statistics_to_doc()

    # Define all queries
    queries = [
        "ان لائن ٹریفک نگرانی نئے حکومتی منصوبے خدشات کراچی حکومت جانب ان لائن ٹریفک ٹیلی کام انڈسٹری ڈیجیٹل حقوق وکالت کرنے والوں نگرانی منصوبے اس عمل شفافیت پرائیویسی حوالے کئی خدشات جنم لے ہیںڈان اخبار رپورٹ پاکستان ٹیلی کمیونیکیشن اتھارٹی پی ٹی اے حال گرے ٹریفک نگرانی تجزیے ٹیلی کام انڈسٹری مناسب تکینکی حل تلاش کرنے احکامات دیے تھے",

        "پاکستان اٹو پارٹس شو ایکسپو سینٹر اپریل شروع ہوگا نمائش پاکستان اٹو سیکٹر ترقی مستقبل کامیابیوں حوالے مواقع اجاگر کرے",

        "پاکستانی اشیا برامدات کیلیے چین صورت نئی مارکیٹ کھل جائے وزارت تجارت ارب ڈالر مالیت مذکورہ پاکستانی پروڈکٹس امریکا یورپی یونین ممالک کوسالانہ بنیادوں برامدکی جاتی",

        "سائنسدان کروڑ سال بے جان جرثوموں زندگی جانب لانے کامیاب سائنسدانوں کامیابی سمندر تہہ ڈائناسور عہد موجود جرثوموں ایک بار زندگی جانب لانے کامیابی حاصل جس بعد جاندار غذا استعمال اپنی تعداد بڑھانے لگے ہیں",

        "ویوو فلیگ شپ ایکس سیریز فونز متعارف ویوو رواں سال فلیگ شپ فونز ایکس سیریز شکل متعارف کرادیئے ہیںاس سیریز فونز خاص بات اس مین کیمرے ہیوی ڈیوٹی اسٹیبلائزیشن سسٹم"
    ]

    # Execute all queries
    for idx, query in enumerate(queries, 1):
        print("\n\n" + "="*70)
        print(f"EXECUTING QUERY {idx} OF {len(queries)}")
        print("="*70)

        results = recommender.get_recommendations_comparison(
            query=query,
            n_results=50
        )

        recommender.compare_and_visualize(results, df=df, query_number=idx)

    # Save the final document
    recommender.save_document()

    print("\n" + "="*70)
    print("✓ COMPARISON SYSTEM DEMO COMPLETED!")
    print("✓ Full embeddings: 768 dimensions")
    print("✓ LDA embeddings: 64 dimensions")
    print(f"✓ All {len(queries)} queries executed")
    print(f"✓ Comprehensive report saved to: {recommender.output_doc_path}")
    print("✓ Visualizations saved as PNG files")
    print("="*70)

URDU NEWS RECOMMENDATION COMPARISON SYSTEM
Full Embeddings (768D) vs LDA Embeddings (64D)
OUTPUTTING TO WORD DOCUMENT

Loading dataset for content display...
Cleaning dataset...
✓ Dataset loaded: 111853 articles

INITIALIZING COMPARISON SYSTEM
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Loading LDA model from: chroma_db_mean_content_LDA64_claude/lda_model.pkl
✓ LDA model loaded (4 dimensions)
Connecting to Full Embeddings ChromaDB at: chroma_db_mean_content_full_claude
✓ Connected to full embeddings collection: urdu_news_embeddings_full_mean_claude
✓ Total articles in full database: 111853
Connecting to LDA Embeddings ChromaDB at: chroma_db_mean_content_LDA64_claude
✓ Connected to LDA embeddings collection: urdu_news_embeddings_lda64_mean_claude
✓ Total articles in LDA database: 111853

SYSTEM STATISTICS
Model: urduhack/roberta-urdu-small
Device: cuda
Pooling Method: MEAN POOLING

Full Embeddings (768D):
  total_articles: 111853
  collection_name: urdu_news_embeddings

# **Auto Encoder DR Technique**

In [11]:
# =============================================================================
# URDU NEWS EMBEDDINGS GENERATION WITH AUTOENCODER DIMENSIONALITY REDUCTION
# Creates two ChromaDB collections: full embeddings (768D) and Autoencoder reduced (64D)
# Using MEAN POOLING - chroma_db_mean_content_full & chroma_db_mean_content_AE64
# =============================================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
import chromadb
from chromadb.config import Settings
from pathlib import Path
import time
import pickle
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class Autoencoder(nn.Module):
    """
    Autoencoder neural network for dimensionality reduction.
    Reduces 768D embeddings to 64D latent space.
    """
    def __init__(self, input_dim=768, latent_dim=64):
        super(Autoencoder, self).__init__()

        # Encoder: 768 -> 384 -> 192 -> 96 -> 64
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 384),
            nn.ReLU(),
            nn.BatchNorm1d(384),
            nn.Dropout(0.2),

            nn.Linear(384, 192),
            nn.ReLU(),
            nn.BatchNorm1d(192),
            nn.Dropout(0.2),

            nn.Linear(192, 96),
            nn.ReLU(),
            nn.BatchNorm1d(96),
            nn.Dropout(0.2),

            nn.Linear(96, latent_dim),
            nn.ReLU()
        )

        # Decoder: 64 -> 96 -> 192 -> 384 -> 768
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 96),
            nn.ReLU(),
            nn.BatchNorm1d(96),
            nn.Dropout(0.2),

            nn.Linear(96, 192),
            nn.ReLU(),
            nn.BatchNorm1d(192),
            nn.Dropout(0.2),

            nn.Linear(192, 384),
            nn.ReLU(),
            nn.BatchNorm1d(384),
            nn.Dropout(0.2),

            nn.Linear(384, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

    def encode(self, x):
        """Return only the encoded (compressed) representation."""
        return self.encoder(x)


class UrduNewsEmbeddingGenerator:
    """
    Generate embeddings for Urdu news articles using UrduHack RoBERTa model
    and store them in two ChromaDB collections:
    1. Full embeddings (768 dimensions)
    2. Autoencoder-reduced embeddings (64 dimensions)

    Designed for large datasets (10,000+ records) with semantic search on content.
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "./chroma_db_mean_content_full_claude",
                 chroma_db_path_ae: str = "./chroma_db_mean_content_AE64_claude",
                 ae_dimensions: int = 64):
        """
        Initialize the embedding generator with model and ChromaDB settings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path_full: Path to store ChromaDB with full embeddings
            chroma_db_path_ae: Path to store ChromaDB with Autoencoder-reduced embeddings
            ae_dimensions: Number of dimensions for Autoencoder reduction (default: 64)
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_ae = Path(chroma_db_path_ae)
        self.ae_dimensions = ae_dimensions
        self.autoencoder = None

        # Create directories
        self.chroma_db_path_full.mkdir(exist_ok=True, parents=True)
        self.chroma_db_path_ae.mkdir(exist_ok=True, parents=True)

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Using device: {self.device}")

        # Load model and tokenizer
        logger.info(f"Loading model: {model_name}")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name).to(self.device)
            self.model.eval()  # Set to evaluation mode
            logger.info("Model loaded successfully")
        except Exception as e:
            logger.error(f"Error loading model: {e}")
            raise

        # Initialize ChromaDB clients for both collections
        logger.info(f"Initializing ChromaDB for full embeddings at: {self.chroma_db_path_full}")
        try:
            self.client_full = chromadb.PersistentClient(
                path=str(self.chroma_db_path_full)
            )
        except Exception as e:
            logger.error(f"Error initializing ChromaDB client for full embeddings: {e}")
            raise

        logger.info(f"Initializing ChromaDB for Autoencoder embeddings at: {self.chroma_db_path_ae}")
        try:
            self.client_ae = chromadb.PersistentClient(
                path=str(self.chroma_db_path_ae)
            )
        except Exception as e:
            logger.error(f"Error initializing ChromaDB client for Autoencoder embeddings: {e}")
            raise

        # Create or get collections for storing embeddings
        try:
            self.collection_full = self.client_full.get_or_create_collection(
                name="urdu_news_embeddings_full_mean_claude",
                metadata={"hnsw:space": "cosine"}  # Use cosine similarity
            )
            logger.info(f"Collection '{self.collection_full.name}' created/retrieved")
        except Exception as e:
            logger.error(f"Error creating full embeddings collection: {e}")
            raise

        try:
            self.collection_ae = self.client_ae.get_or_create_collection(
                name="urdu_news_embeddings_ae64_mean_claude",
                metadata={"hnsw:space": "cosine"}  # Use cosine similarity
            )
            logger.info(f"Collection '{self.collection_ae.name}' created/retrieved")
        except Exception as e:
            logger.error(f"Error creating Autoencoder embeddings collection: {e}")
            raise

    def mean_pooling(self, model_output, attention_mask):
        """
        Apply mean pooling to model output to get sentence embeddings.

        Args:
            model_output: Output from transformer model
            attention_mask: Attention mask from tokenizer

        Returns:
            Mean pooled embeddings (batch_size, embedding_dim)
        """
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_embedding_for_text(self, text: str, max_length: int = 512,
                                    chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate embedding for a single text using mean pooling.

        Args:
            text: Input Urdu text (content column)
            max_length: Maximum tokens per chunk (default: 512)
            chunk_overlap: Overlap between chunks to maintain context (default: 50)

        Returns:
            Embedding vector as numpy array (768,)
        """
        if not text or pd.isna(text) or str(text).strip() == "":
            logger.warning("Empty text provided for embedding generation")
            return np.zeros(768)

        text = str(text)
        
        tokens = self.tokenizer.encode(text, add_special_tokens=True)

        if len(tokens) <= max_length:
            try:
                encoded_input = self.tokenizer(
                    text,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors='pt'
                )
                encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

                with torch.no_grad():
                    model_output = self.model(**encoded_input)

                embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
                embeddings = embeddings.cpu().detach().numpy()
                return embeddings[0]
            except Exception as e:
                logger.error(f"Error generating embedding: {e}")
                return np.zeros(768)

        # For long articles: Split into overlapping chunks
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            try:
                encoded_input = self.tokenizer(
                    chunk_text,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors='pt'
                )
                encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

                with torch.no_grad():
                    model_output = self.model(**encoded_input)

                chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
                chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])
            except Exception as e:
                logger.warning(f"Error processing chunk {i}: {e}")
                continue

        if len(chunk_embeddings) == 0:
            logger.warning("No valid chunks processed, returning zero vector")
            return np.zeros(768)

        final_embedding = np.mean(chunk_embeddings, axis=0)
        return final_embedding

    def train_autoencoder(self, embeddings_array: np.ndarray,
                         epochs: int = 50,
                         batch_size: int = 256,
                         learning_rate: float = 0.001) -> None:
        """
        Train autoencoder on the full embeddings.

        Args:
            embeddings_array: Array of shape (n_samples, embedding_dim)
            epochs: Number of training epochs
            batch_size: Batch size for training
            learning_rate: Learning rate for optimizer
        """
        logger.info(f"\n{'='*70}")
        logger.info(f"TRAINING AUTOENCODER MODEL ({self.ae_dimensions} dimensions)")
        logger.info(f"{'='*70}")
        logger.info(f"Input shape: {embeddings_array.shape}")
        logger.info(f"Epochs: {epochs}")
        logger.info(f"Batch size: {batch_size}")
        logger.info(f"Learning rate: {learning_rate}")

        # Initialize autoencoder
        input_dim = embeddings_array.shape[1]
        self.autoencoder = Autoencoder(input_dim=input_dim, latent_dim=self.ae_dimensions).to(self.device)

        # Prepare data
        X_tensor = torch.FloatTensor(embeddings_array)
        dataset = TensorDataset(X_tensor)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        # Loss and optimizer
        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.autoencoder.parameters(), lr=learning_rate)

        # Training loop
        logger.info("\nTraining progress:")
        logger.info("-" * 70)

        for epoch in range(epochs):
            epoch_loss = 0.0
            num_batches = 0

            self.autoencoder.train()
            for batch in dataloader:
                X_batch = batch[0].to(self.device)

                # Forward pass
                reconstructed = self.autoencoder(X_batch)
                loss = criterion(reconstructed, X_batch)

                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                num_batches += 1

            avg_loss = epoch_loss / num_batches

            if (epoch + 1) % 5 == 0 or epoch == 0:
                logger.info(f"Epoch [{epoch+1:3d}/{epochs}] - Loss: {avg_loss:.6f}")

        logger.info("-" * 70)
        logger.info(f"✓ Autoencoder training completed!")
        logger.info(f"Final reconstruction loss: {avg_loss:.6f}")

        # Save autoencoder model
        self.autoencoder.eval()
        ae_model_path = self.chroma_db_path_ae / "autoencoder_model.pkl"
        torch.save({
            'model_state_dict': self.autoencoder.state_dict(),
            'input_dim': input_dim,
            'latent_dim': self.ae_dimensions,
            'final_loss': avg_loss
        }, ae_model_path)
        logger.info(f"✓ Autoencoder model saved to: {ae_model_path}")

    def apply_autoencoder(self, embeddings_array: np.ndarray) -> np.ndarray:
        """
        Apply autoencoder transformation to embeddings.

        Args:
            embeddings_array: Array of shape (n_samples, embedding_dim)

        Returns:
            Reduced embeddings of shape (n_samples, ae_dimensions)
        """
        if self.autoencoder is None:
            raise ValueError("Autoencoder not trained. Call train_autoencoder first.")

        self.autoencoder.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(embeddings_array).to(self.device)
            encoded = self.autoencoder.encode(X_tensor)
            return encoded.cpu().numpy()

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       content_column: str = "content",
                                       headline_column: str = "Headline",
                                       category_column: str = "Category") -> None:
        """
        Generate embeddings for all articles and store in two ChromaDB collections:
        1. Full embeddings (768D)
        2. Autoencoder-reduced embeddings (64D)

        Args:
            df: Dataframe containing articles
            content_column: Column containing article content
            headline_column: Column containing article headline
            category_column: Column containing article category
        """
        logger.info(f"\n{'='*70}")
        logger.info(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        logger.info(f"{'='*70}")
        logger.info(f"Content column: '{content_column}' (used for semantic search)")
        logger.info(f"Headline column: '{headline_column}' (stored as metadata)")
        logger.info(f"Category column: '{category_column}' (stored as metadata)")
        logger.info(f"Pooling method: MEAN POOLING")
        logger.info(f"Full embeddings DB: chroma_db_mean_content_full")
        logger.info(f"Autoencoder embeddings DB: chroma_db_mean_content_AE64")
        logger.info(f"{'='*70}\n")

        total_articles = len(df)
        start_time = time.time()

        # Prepare data
        ids = []
        embeddings_full = []
        metadatas = []
        documents = []

        # Step 1: Generate full embeddings
        logger.info("STEP 1: Generating full embeddings (768D)...")
        logger.info("="*70)

        successful_articles = 0
        failed_articles = 0
        empty_content_count = 0

        for idx, row in df.iterrows():
            if (idx + 1) % 10000 == 0:
                elapsed = time.time() - start_time
                articles_per_sec = (idx + 1) / elapsed if elapsed > 0 else 0
                eta = (total_articles - idx - 1) / articles_per_sec if articles_per_sec > 0 else 0
                progress_msg = f"Processed {idx + 1}/{total_articles} articles ({elapsed:.2f}s elapsed, ETA: {eta:.2f}s)"
                logger.info(progress_msg)

            content_text = str(row[content_column]) if not pd.isna(row[content_column]) else ""

            if len(content_text.strip()) == 0:
                empty_content_count += 1
                if empty_content_count <= 5:
                    logger.warning(f"Skipping article {idx} - empty content")
                continue

            try:
                # Generate full embedding
                embedding = self.generate_embedding_for_text(content_text)

                doc_id = f"article_{idx}"
                ids.append(doc_id)
                embeddings_full.append(embedding)
                documents.append(content_text[:500])

                headline = str(row.get(headline_column, "Unknown")) if not pd.isna(row.get(headline_column)) else "Unknown"
                category = str(row.get(category_column, "Unknown")) if not pd.isna(row.get(category_column)) else "Unknown"
                
                metadata = {
                    "article_index": idx,
                    "headline": headline,
                    "category": category,
                    "content_length": len(content_text),
                    "pooling_method": "mean_pooling"
                }
                metadatas.append(metadata)
                successful_articles += 1

            except Exception as e:
                failed_articles += 1
                error_msg = f"Error processing article {idx}: {str(e)[:100]}"
                if failed_articles <= 5:
                    logger.error(error_msg)
                continue

        # Log final progress
        if total_articles % 10000 != 0:
            elapsed = time.time() - start_time
            progress_msg = f"Processed {total_articles}/{total_articles} articles ({elapsed:.2f}s elapsed)"
            logger.info(progress_msg)

        logger.info(f"\nStep 1 Summary:")
        logger.info(f"  Total articles processed: {total_articles}")
        logger.info(f"  Successfully embedded: {successful_articles}")
        logger.info(f"  Failed: {failed_articles}")
        logger.info(f"  Empty content skipped: {empty_content_count}")

        # Convert to numpy array for Autoencoder
        embeddings_full_array = np.array(embeddings_full)

        # Step 2: Train and apply Autoencoder
        logger.info(f"\n{'='*70}")
        logger.info("STEP 2: Training Autoencoder...")
        logger.info("="*70)
        
        self.train_autoencoder(embeddings_full_array, epochs=50, batch_size=256)

        logger.info(f"\n{'='*70}")
        logger.info("STEP 3: Applying Autoencoder transformation...")
        logger.info("="*70)

        embeddings_ae = self.apply_autoencoder(embeddings_full_array)
        logger.info(f"✓ Autoencoder transformation complete!")
        logger.info(f"Reduced embeddings shape: {embeddings_ae.shape}")

        # Step 3: Store full embeddings in ChromaDB
        logger.info(f"\n{'='*70}")
        logger.info(f"STEP 4: STORING FULL EMBEDDINGS (768D) IN CHROMADB...")
        logger.info(f"{'='*70}")

        batch_size = 5000
        total_batches = (len(ids) + batch_size - 1) // batch_size

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            logger.info(f"Storing batch {current_batch}/{total_batches} "
                      f"(items {batch_idx} to {batch_end})...")

            # Add metadata indicating this is full embedding
            batch_metadatas = [
                {**meta, "embedding_type": "full", "dimensions": 768}
                for meta in metadatas[batch_idx:batch_end]
            ]

            self.collection_full.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings_full[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )

        logger.info(f"✓ Full embeddings stored successfully!")

        # Step 4: Store Autoencoder embeddings in ChromaDB
        logger.info(f"\n{'='*70}")
        logger.info(f"STEP 5: STORING AUTOENCODER EMBEDDINGS (64D) IN CHROMADB...")
        logger.info(f"{'='*70}")

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            current_batch = (batch_idx // batch_size) + 1

            logger.info(f"Storing batch {current_batch}/{total_batches} "
                      f"(items {batch_idx} to {batch_end})...")

            # Add metadata indicating this is Autoencoder embedding
            batch_metadatas = [
                {**meta, "embedding_type": "autoencoder", "dimensions": self.ae_dimensions}
                for meta in metadatas[batch_idx:batch_end]
            ]

            self.collection_ae.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings_ae[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )

        logger.info(f"✓ Autoencoder embeddings stored successfully!")

        total_time = time.time() - start_time
        logger.info(f"\n{'='*70}")
        logger.info(f"✓ EMBEDDING GENERATION COMPLETE!")
        logger.info(f"{'='*70}")
        logger.info(f"Total embeddings stored: {len(ids)}")
        logger.info(f"Full embeddings (768D): {len(ids)} in chroma_db_mean_content_full")
        logger.info(f"Autoencoder embeddings (64D): {len(ids)} in chroma_db_mean_content_AE64")
        logger.info(f"Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
        logger.info(f"Average time per article: {total_time/len(ids):.4f} seconds" if len(ids) > 0 else "No articles processed")
        logger.info(f"Processing speed: {len(ids)/total_time:.2f} articles/second" if total_time > 0 and len(ids) > 0 else "N/A")

    def search_similar_articles(self, query_text: str, n_results: int = 5,
                               use_ae: bool = False) -> dict:
        """
        Search for similar articles using query text.

        Args:
            query_text: Query text to find similar articles
            n_results: Number of similar articles to return
            use_ae: If True, search in Autoencoder collection; if False, search in full collection

        Returns:
            Dictionary containing similar articles and their distances
        """
        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Apply Autoencoder if needed
        if use_ae:
            if self.autoencoder is None:
                # Try to load Autoencoder model
                ae_model_path = self.chroma_db_path_ae / "autoencoder_model.pkl"
                if ae_model_path.exists():
                    checkpoint = torch.load(ae_model_path)
                    self.autoencoder = Autoencoder(
                        input_dim=checkpoint['input_dim'],
                        latent_dim=checkpoint['latent_dim']
                    ).to(self.device)
                    self.autoencoder.load_state_dict(checkpoint['model_state_dict'])
                    self.autoencoder.eval()
                else:
                    raise ValueError("Autoencoder model not found. Generate embeddings first.")

            query_embedding = self.apply_autoencoder(query_embedding.reshape(1, -1))[0]
            collection = self.collection_ae
        else:
            collection = self.collection_full

        # Search in ChromaDB
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )

        return results

    def get_collection_stats(self) -> dict:
        """
        Get statistics about both ChromaDB collections.

        Returns:
            Dictionary with collection information
        """
        count_full = self.collection_full.count()
        count_ae = self.collection_ae.count()

        return {
            "full_embeddings": {
                "total_embeddings": count_full,
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768,
                "db_path": str(self.chroma_db_path_full)
            },
            "autoencoder_embeddings": {
                "total_embeddings": count_ae,
                "collection_name": self.collection_ae.name,
                "embedding_dimension": self.ae_dimensions,
                "db_path": str(self.chroma_db_path_ae),
                "model_type": "Neural Network Autoencoder"
            },
            "pooling_method": "MEAN POOLING",
            "search_column": "content"
        }


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS EMBEDDINGS WITH AUTOENCODER DIMENSIONALITY REDUCTION")
    print("Full Embeddings: chroma_db_mean_content_full (768D)")
    print("Autoencoder Embeddings: chroma_db_mean_content_AE64 (64D)")
    print("="*70)
    print("\nLoading dataset...")

    try:
        df = pd.read_csv("final_cleaned_urdu_news.csv")
        
        # Clean the data - handle NaN values in key columns
        print("Cleaning dataset...")
        df['Category'] = df['Category'].fillna('Unknown')
        df['Headline'] = df['Headline'].fillna('Unknown')
        df['content'] = df['content'].fillna('')
        
        print(f"\nDataset loaded successfully!")
        print(f"Dataset shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        
        # Get unique categories, handle NaN values
        categories = df['Category'].unique().tolist()
        categories_str = [str(cat) if pd.notna(cat) else "Unknown" for cat in categories]
        
        print(f"\nDataset info:")
        print(f"  - Total articles: {len(df)}")
        print(f"  - Unique categories: {df['Category'].nunique()}")
        print(f"  - Categories sample: {', '.join(categories_str[:10])}..." if len(categories_str) > 10 else f"  - Categories: {', '.join(categories_str)}")

        # Show sample data
        print(f"\n{'='*70}")
        print("SAMPLE DATA PREVIEW")
        print(f"{'='*70}")
        sample = df.iloc[0]
        print(f"Headline: {str(sample['Headline'])[:100]}...")
        print(f"Category: {sample['Category']}")
        print(f"Content preview: {str(sample['content'])[:200]}...")
        print(f"Content length: {len(str(sample['content']))} characters")

        # Initialize embedding generator
        print(f"\n{'='*70}")
        print("INITIALIZING EMBEDDING GENERATOR")
        print(f"{'='*70}")

        embedder = UrduNewsEmbeddingGenerator(
            model_name="urduhack/roberta-urdu-small",
            chroma_db_path_full="./chroma_db_mean_content_full_claude",
            chroma_db_path_ae="./chroma_db_mean_content_AE64_claude",
            ae_dimensions=64
        )

        # Generate embeddings and store in both ChromaDB collections
        embedder.generate_embeddings_for_dataset(
            df=df,
            content_column="content",
            headline_column="Headline",
            category_column="Category"
        )

        # Display collection statistics
        print("\n" + "="*70)
        print("CHROMADB COLLECTION STATISTICS")
        print("="*70)
        stats = embedder.get_collection_stats()
        print("\nFULL EMBEDDINGS COLLECTION:")
        for key, value in stats["full_embeddings"].items():
            print(f"  {key}: {value}")
        print("\nAUTOENCODER EMBEDDINGS COLLECTION:")
        for key, value in stats["autoencoder_embeddings"].items():
            print(f"  {key}: {value}")
        print(f"\nPooling method: {stats['pooling_method']}")
        print(f"Search column: {stats['search_column']}")

        # Test: Search in both collections
        print("\n" + "="*70)
        print("TESTING SEMANTIC SEARCH")
        print("="*70)

        query = "پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن"
        print(f"\nQuery: {query}\n")

        # Search in full embeddings
        print("="*70)
        print("RESULTS FROM FULL EMBEDDINGS (768D)")
        print("="*70)
        results_full = embedder.search_similar_articles(query_text=query, n_results=5, use_ae=False)

        for i, (doc_id, distance, metadata) in enumerate(zip(
            results_full['ids'][0],
            results_full['distances'][0],
            results_full['metadatas'][0]
        )):
            print(f"\nResult #{i+1}")
            print(f"  Similarity Score: {1 - distance:.4f}")
            print(f"  Headline: {metadata.get('headline', 'N/A')[:80]}...")
            print(f"  Category: {metadata.get('category', 'N/A')}")

        # Search in Autoencoder embeddings
        print("\n" + "="*70)
        print("RESULTS FROM AUTOENCODER EMBEDDINGS (64D)")
        print("="*70)
        results_ae = embedder.search_similar_articles(query_text=query, n_results=5, use_ae=True)

        for i, (doc_id, distance, metadata) in enumerate(zip(
            results_ae['ids'][0],
            results_ae['distances'][0],
            results_ae['metadatas'][0]
        )):
            print(f"\nResult #{i+1}")
            print(f"  Similarity Score: {1 - distance:.4f}")
            print(f"  Headline: {metadata.get('headline', 'N/A')[:80]}...")
            print(f"  Category: {metadata.get('category', 'N/A')}")

        print("\n" + "="*70)
        print("✓ PROCESSING COMPLETED SUCCESSFULLY!")
        print("✓ Two ChromaDB collections created:")
        print("  1. chroma_db_mean_content_full (768D)")
        print("  2. chroma_db_mean_content_AE64 (64D)")
        print("="*70)

    except FileNotFoundError as e:
        print(f"\n❌ ERROR: Could not find the dataset file: {e}")
        print("Please make sure 'balanced_dataset.csv' exists in the current directory.")
    except Exception as e:
        print(f"\n❌ ERROR: An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()

URDU NEWS EMBEDDINGS WITH AUTOENCODER DIMENSIONALITY REDUCTION
Full Embeddings: chroma_db_mean_content_full (768D)
Autoencoder Embeddings: chroma_db_mean_content_AE64 (64D)

Loading dataset...


2026-01-17 20:25:54,042 - INFO - Using device: cuda
2026-01-17 20:25:54,047 - INFO - Loading model: urduhack/roberta-urdu-small


Cleaning dataset...

Dataset loaded successfully!
Dataset shape: (111853, 3)
Columns: ['Headline', 'Category', 'content']

Dataset info:
  - Total articles: 111853
  - Unique categories: 5
  - Categories: Business & Economics, Entertainment, Science & Technology, Sports, Unknown

SAMPLE DATA PREVIEW
Headline: عالمی بینک عسکریت پسندی سے متاثرہ خاندانوں کی معاونت کرے گا...
Category: Business & Economics
Content preview: عالمی بینک عسکریت پسندی متاثرہ خاندانوں معاونت کرے اسلام باد عالمی بینک خیبرپختونخوا قبائلی اضلاع عسکریت پسندی پیدا ہونے بحران متاثرہ خاندانوں جلد بحالی بچوں صحت بہتری شہری مراکز ترسیل معاونت فنڈز فرا...
Content length: 1504 characters

INITIALIZING EMBEDDING GENERATOR


2026-01-17 20:25:56,200 - INFO - Model loaded successfully
2026-01-17 20:25:56,202 - INFO - Initializing ChromaDB for full embeddings at: chroma_db_mean_content_full_claude
2026-01-17 20:25:56,205 - INFO - Initializing ChromaDB for Autoencoder embeddings at: chroma_db_mean_content_AE64_claude
2026-01-17 20:25:56,207 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-01-17 20:25:56,262 - INFO - Collection 'urdu_news_embeddings_full_mean_claude' created/retrieved
2026-01-17 20:25:56,268 - INFO - Collection 'urdu_news_embeddings_ae64_mean_claude' created/retrieved
2026-01-17 20:25:56,288 - INFO - 
2026-01-17 20:25:56,294 - INFO - GENERATING EMBEDDINGS FOR 111853 ARTICLES
2026-01-17 20:25:56,298 - INFO - ======================================================================
2026-01-17 20:25:56,301 - INFO - Content column: 'content' (used for semantic search)
2026-01-17 20:25:56,302 - INFO - Headline column: 'Headli


CHROMADB COLLECTION STATISTICS

FULL EMBEDDINGS COLLECTION:
  total_embeddings: 111853
  collection_name: urdu_news_embeddings_full_mean_claude
  embedding_dimension: 768
  db_path: chroma_db_mean_content_full_claude

AUTOENCODER EMBEDDINGS COLLECTION:
  total_embeddings: 111853
  collection_name: urdu_news_embeddings_ae64_mean_claude
  embedding_dimension: 64
  db_path: chroma_db_mean_content_AE64_claude
  model_type: Neural Network Autoencoder

Pooling method: MEAN POOLING
Search column: content

TESTING SEMANTIC SEARCH

Query: پاکستان میں موبائل کمپنیاں مقامی طور پر اسمبلنگ کی جانب گامزن

RESULTS FROM FULL EMBEDDINGS (768D)

Result #1
  Similarity Score: 0.8087
  Headline: لکڑی سے کوئلہ تک...
  Category: Entertainment

Result #2
  Similarity Score: 0.8026
  Headline: کراچی جاپان ایکسٹرنل ٹریڈ رگنائزیشن کی جانب سے تین روزہ نمائش شروع...
  Category: Business & Economics

Result #3
  Similarity Score: 0.8019
  Headline: جاپان میں ماحول دوست گاڑیوں کا میلہ...
  Category: Entertainment


# **Recommender Code AutoEncoder**

In [3]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM WITH FULL VS AUTOENCODER EMBEDDINGS COMPARISON
# Compares recommendations from 768D full embeddings and 64D Autoencoder-reduced embeddings
# OUTPUTS RESULTS TO WORD DOCUMENT
# =============================================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure matplotlib for better visualization
rcParams['figure.figsize'] = (16, 10)
rcParams['font.size'] = 10
sns.set_style("whitegrid")


class Autoencoder(nn.Module):
    """
    Autoencoder neural network for dimensionality reduction.
    Reduces 768D embeddings to 64D latent space.
    """
    def __init__(self, input_dim=768, latent_dim=64):
        super(Autoencoder, self).__init__()

        # Encoder: 768 -> 384 -> 192 -> 96 -> 64
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 384),
            nn.ReLU(),
            nn.BatchNorm1d(384),
            nn.Dropout(0.2),

            nn.Linear(384, 192),
            nn.ReLU(),
            nn.BatchNorm1d(192),
            nn.Dropout(0.2),

            nn.Linear(192, 96),
            nn.ReLU(),
            nn.BatchNorm1d(96),
            nn.Dropout(0.2),

            nn.Linear(96, latent_dim),
            nn.ReLU()
        )

        # Decoder: 64 -> 96 -> 192 -> 384 -> 768
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 96),
            nn.ReLU(),
            nn.BatchNorm1d(96),
            nn.Dropout(0.2),

            nn.Linear(96, 192),
            nn.ReLU(),
            nn.BatchNorm1d(192),
            nn.Dropout(0.2),

            nn.Linear(192, 384),
            nn.ReLU(),
            nn.BatchNorm1d(384),
            nn.Dropout(0.2),

            nn.Linear(384, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

    def encode(self, x):
        """Return only the encoded (compressed) representation."""
        return self.encoder(x)


class UrduNewsRecommenderComparison:
    """
    Recommendation system that compares results from full embeddings (768D)
    and Autoencoder-reduced embeddings (64D).
    Outputs results to Word document.
    """

    def __init__(self,
                 model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path_full: str = "./chroma_db_mean_content_full_claude",
                 chroma_db_path_ae: str = "./chroma_db_mean_content_AE64_claude",
                 collection_name_full: str = "urdu_news_embeddings_full_mean_claude",
                 collection_name_ae: str = "urdu_news_embeddings_ae64_mean_claude",
                 output_doc_path: str = "Urdu_News_Full_vs_Autoencoder_Comparison_Report.docx"):
        """
        Initialize recommender with connections to both full and Autoencoder databases.
        """
        self.model_name = model_name
        self.chroma_db_path_full = Path(chroma_db_path_full)
        self.chroma_db_path_ae = Path(chroma_db_path_ae)
        self.autoencoder = None
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Load Autoencoder model
        ae_model_path = self.chroma_db_path_ae / "autoencoder_model.pkl"
        if ae_model_path.exists():
            print(f"Loading Autoencoder model from: {ae_model_path}")
            self.add_paragraph(f"Loading Autoencoder model from: {ae_model_path}")
            checkpoint = torch.load(ae_model_path, map_location=self.device)
            self.autoencoder = Autoencoder(
                input_dim=checkpoint['input_dim'],
                latent_dim=checkpoint['latent_dim']
            ).to(self.device)
            self.autoencoder.load_state_dict(checkpoint['model_state_dict'])
            self.autoencoder.eval()
            ae_dimensions = checkpoint.get('latent_dim', 64)
            print(f"✓ Autoencoder model loaded ({ae_dimensions} dimensions)")
            print(f"  Final training loss: {checkpoint.get('final_loss', 'N/A')}")
            self.add_paragraph(f"✓ Autoencoder model loaded ({ae_dimensions} dimensions)")
            self.add_paragraph(f"  Final training loss: {checkpoint.get('final_loss', 'N/A')}")
        else:
            raise FileNotFoundError(f"Autoencoder model not found at {ae_model_path}")

        # Connect to full embeddings ChromaDB
        print(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.add_paragraph(f"Connecting to Full Embeddings ChromaDB at: {self.chroma_db_path_full}")
        self.client_full = chromadb.PersistentClient(path=str(self.chroma_db_path_full))

        try:
            self.collection_full = self.client_full.get_collection(name=collection_name_full)
            print(f"✓ Connected to full embeddings collection: {collection_name_full}")
            print(f"✓ Total articles in full database: {self.collection_full.count()}")
            self.add_paragraph(f"✓ Connected to full embeddings collection: {collection_name_full}")
            self.add_paragraph(f"✓ Total articles in full database: {self.collection_full.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_full}'")
            raise e

        # Connect to Autoencoder embeddings ChromaDB
        print(f"Connecting to Autoencoder Embeddings ChromaDB at: {self.chroma_db_path_ae}")
        self.add_paragraph(f"Connecting to Autoencoder Embeddings ChromaDB at: {self.chroma_db_path_ae}")
        self.client_ae = chromadb.PersistentClient(path=str(self.chroma_db_path_ae))

        try:
            self.collection_ae = self.client_ae.get_collection(name=collection_name_ae)
            print(f"✓ Connected to Autoencoder embeddings collection: {collection_name_ae}")
            print(f"✓ Total articles in Autoencoder database: {self.collection_ae.count()}")
            self.add_paragraph(f"✓ Connected to Autoencoder embeddings collection: {collection_name_ae}")
            self.add_paragraph(f"✓ Total articles in Autoencoder database: {self.collection_ae.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name_ae}'")
            raise e

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System\nFull (768D) vs Autoencoder (64D) Comparison Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text='', style=None):
        """Add paragraph to document."""
        para = self.doc.add_paragraph(text, style)
        return para

    def add_table(self, data, headers=None):
        """Add table to document."""
        if headers:
            table = self.doc.add_table(rows=1, cols=len(headers))
            table.style = 'Light Grid Accent 1'
            header_cells = table.rows[0].cells
            for i, header in enumerate(headers):
                header_cells[i].text = str(header)
            
            # Add data rows
            for row_data in data:
                cells = table.add_row().cells
                for col_idx, cell_data in enumerate(row_data):
                    cells[col_idx].text = str(cell_data)
        else:
            table = self.doc.add_table(rows=len(data), cols=len(data[0]))
            table.style = 'Light Grid Accent 1'
            
            for row_idx, row_data in enumerate(data):
                cells = table.rows[row_idx].cells
                for col_idx, cell_data in enumerate(row_data):
                    cells[col_idx].text = str(cell_data)

        return table

    def add_image(self, image_path, width_inches=6):
        """Add image to document."""
        try:
            self.doc.add_picture(image_path, width=Inches(width_inches))
            last_paragraph = self.doc.paragraphs[-1]
            last_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
            self.add_paragraph('')  # Add empty line after image
        except Exception as e:
            self.add_paragraph(f"Error adding image: {str(e)}")

    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"Word document saved to: {self.output_doc_path}")

    def mean_pooling(self, model_output, attention_mask):
        """Apply MEAN POOLING to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50, apply_ae: bool = False) -> np.ndarray:
        """
        Generate embedding for query text using MEAN POOLING.
        Optionally applies Autoencoder dimensionality reduction.
        """
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # Process short queries
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embedding = embeddings.cpu().detach().numpy()[0]

            # Apply Autoencoder if requested
            if apply_ae and self.autoencoder is not None:
                with torch.no_grad():
                    embedding_tensor = torch.FloatTensor(embedding).unsqueeze(0).to(self.device)
                    encoded = self.autoencoder.encode(embedding_tensor)
                    embedding = encoded.cpu().numpy()[0]

            return embedding

        # For long queries: use chunking
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)

        # Apply Autoencoder if requested
        if apply_ae and self.autoencoder is not None:
            with torch.no_grad():
                embedding_tensor = torch.FloatTensor(final_embedding).unsqueeze(0).to(self.device)
                encoded = self.autoencoder.encode(embedding_tensor)
                final_embedding = encoded.cpu().numpy()[0]

        return final_embedding

    def get_recommendations_comparison(self, query: str, n_results: int = 10,
                                      filter_category: str = None) -> dict:
        """
        Get recommendations from both full and Autoencoder embeddings for comparison.
        """
        print(f"\n{'='*70}")
        print(f"GENERATING RECOMMENDATIONS - FULL vs AUTOENCODER COMPARISON")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*70}\n")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Generate embeddings for both spaces
        print("→ Generating query embedding (768D - Full)...")
        query_embedding_full = self.generate_query_embedding(query, apply_ae=False)
        print(f"  ✓ Full embedding generated: shape {query_embedding_full.shape}")

        print("→ Generating query embedding (64D - Autoencoder)...")
        query_embedding_ae = self.generate_query_embedding(query, apply_ae=True)
        print(f"  ✓ Autoencoder embedding generated: shape {query_embedding_ae.shape}")

        # Search in full embeddings
        print("\n→ Searching in Full Embeddings (768D)...")
        results_full = self.collection_full.query(
            query_embeddings=[query_embedding_full.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results_full['ids'][0])} recommendations")

        # Search in Autoencoder embeddings
        print("→ Searching in Autoencoder Embeddings (64D)...")
        results_ae = self.collection_ae.query(
            query_embeddings=[query_embedding_ae.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results_ae['ids'][0])} recommendations\n")

        return {
            'full': results_full,
            'ae': results_ae,
            'query': query,
            'n_results': n_results
        }

    def compare_and_visualize(self, comparison_results: dict, df: pd.DataFrame = None, query_number: int = 1):
        """
        Compare and visualize recommendations from full vs Autoencoder embeddings.
        Adds results to Word document.
        """
        results_full = comparison_results['full']
        results_ae = comparison_results['ae']
        query = comparison_results['query']
        n_results = comparison_results.get('n_results', 10)

        if not results_full['ids'] or not results_ae['ids']:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return

        # Add query heading to document
        self.add_heading(f"Query {query_number}: Comparison Results", level=1)
        self.add_paragraph(f"Query: {query[:200]}...")
        self.add_paragraph(f"Top {n_results} results from each method")
        self.add_paragraph('')

        # Extract data
        full_ids = results_full['ids'][0]
        full_distances = results_full['distances'][0]
        full_metadatas = results_full['metadatas'][0]

        ae_ids = results_ae['ids'][0]
        ae_distances = results_ae['distances'][0]
        ae_metadatas = results_ae['metadatas'][0]

        # Convert distances to similarity scores
        full_similarities = [1 - d for d in full_distances]
        ae_similarities = [1 - d for d in ae_distances]

        # Track overlap between results
        full_set = set(full_ids)
        ae_set = set(ae_ids)
        overlap = full_set.intersection(ae_set)

        # Print and add to doc: comparison summary
        print(f"\n{'='*70}")
        print(f"RECOMMENDATION COMPARISON SUMMARY - Query {query_number}")
        print(f"{'='*70}")
        print(f"Query: {query[:100]}...\n")
        print(f"Total recommendations per method: {n_results}")
        print(f"Articles in common: {len(overlap)} ({len(overlap)/n_results*100:.1f}%)")
        print(f"Unique to Full (768D): {len(full_set - ae_set)}")
        print(f"Unique to Autoencoder (64D): {len(ae_set - full_set)}")

        self.add_heading("Comparison Summary", level=2)
        summary_data = [
            ["Total recommendations per method", n_results],
            ["Articles in common", f"{len(overlap)} ({len(overlap)/n_results*100:.1f}%)"],
            ["Unique to Full (768D)", len(full_set - ae_set)],
            ["Unique to Autoencoder (64D)", len(ae_set - full_set)]
        ]
        self.add_table(summary_data, headers=["Metric", "Value"])
        self.add_paragraph('')

        # Create side-by-side comparison table for document
        self.add_heading("Side-by-Side Comparison", level=2)
        comparison_table_data = []

        for i in range(n_results):
            rank = i + 1

            # Full embedding info
            if i < len(full_ids):
                full_headline = full_metadatas[i].get('headline', 'N/A')[:80]
                full_sim = f"{full_similarities[i]:.4f}"
            else:
                full_headline = "N/A"
                full_sim = "0.0000"

            # Autoencoder embedding info
            if i < len(ae_ids):
                ae_headline = ae_metadatas[i].get('headline', 'N/A')[:80]
                ae_sim = f"{ae_similarities[i]:.4f}"
            else:
                ae_headline = "N/A"
                ae_sim = "0.0000"

            comparison_table_data.append([rank, full_headline, full_sim, ae_headline, ae_sim])

        self.add_table(comparison_table_data,
                      headers=["Rank", "Full (768D) Headline", "Full Similarity",
                              "Autoencoder (64D) Headline", "AE Similarity"])
        self.add_paragraph('')

        # Create and add visualization
        image_path = self._create_visualizations(overlap, full_set, ae_set, query, n_results, query_number)

        self.add_heading("Overlap Visualization", level=2)
        if image_path:
            self.add_image(image_path, width_inches=5)
        self.add_paragraph('')

    def _create_visualizations(self, overlap, full_set, ae_set, query, n_results, query_number):
        """Create comparison visualizations and return image path."""
        fig = plt.figure(figsize=(8, 6))

        # Overlap Analysis
        ax = plt.subplot(1, 1, 1)

        overlap_count = len(overlap)
        full_only = len(full_set - ae_set)
        ae_only = len(ae_set - full_set)

        sizes = [overlap_count, full_only, ae_only]
        labels = [f'Both\n({overlap_count})', f'Full Only\n({full_only})', f'AE Only\n({ae_only})']
        colors_pie = ['#4CAF50', '#2E86AB', '#E63946']
        explode = (0.1, 0, 0)

        ax.pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 10})
        ax.set_title('Recommendation Overlap: Full (768D) vs Autoencoder (64D)',
                     fontsize=12, fontweight='bold', pad=15)

        # Overall title
        fig.suptitle(f'Query {query_number} - Recommendation Overlap\nTop {n_results} Results',
                    fontsize=14, fontweight='bold', y=0.995)

        plt.tight_layout(rect=[0, 0.03, 1, 0.985])

        image_path = f'recommendation_comparison_autoencoder_query_{query_number}.png'
        plt.savefig(image_path, dpi=300, bbox_inches='tight')
        print(f"\n✓ Visualization saved as '{image_path}'")
        plt.close()

        return image_path

    def get_statistics(self) -> dict:
        """Get statistics about both recommendation systems."""
        ae_dimensions = 64  # Fixed to 64D based on specification
        
        return {
            "model": self.model_name,
            "device": str(self.device),
            "full_embeddings": {
                "total_articles": self.collection_full.count(),
                "collection_name": self.collection_full.name,
                "embedding_dimension": 768
            },
            "autoencoder_embeddings": {
                "total_articles": self.collection_ae.count(),
                "collection_name": self.collection_ae.name,
                "embedding_dimension": ae_dimensions,
                "model_type": "Neural Network Autoencoder",
                "architecture": "768→384→192→96→64→96→192→384→768"
            },
            "pooling_method": "MEAN POOLING"
        }

    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        stats = self.get_statistics()

        stats_data = [
            ["Model", stats['model']],
            ["Device", stats['device']],
            ["Pooling Method", stats['pooling_method']]
        ]
        self.add_table(stats_data, headers=["Parameter", "Value"])

        self.add_heading("Full Embeddings (768D)", level=2)
        full_stats = [[k, v] for k, v in stats['full_embeddings'].items()]
        self.add_table(full_stats, headers=["Metric", "Value"])

        self.add_heading("Autoencoder Embeddings (64D)", level=2)
        ae_stats = [[k, v] for k, v in stats['autoencoder_embeddings'].items()]
        self.add_table(ae_stats, headers=["Metric", "Value"])
        self.add_paragraph('')


# =============================================================================
# MAIN EXECUTION - COMPARISON SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("URDU NEWS RECOMMENDATION COMPARISON SYSTEM")
    print("Full Embeddings (768D) vs Autoencoder Embeddings (64D)")
    print("OUTPUTTING TO WORD DOCUMENT")
    print("="*70)

    # Load dataset
    print("\nLoading dataset for content display...")
    try:
        df = pd.read_csv("final_cleaned_urdu_news.csv")
        
        # Clean the data - handle NaN values in key columns
        print("Cleaning dataset...")
        df['Category'] = df['Category'].fillna('Unknown')
        df['Headline'] = df['Headline'].fillna('Unknown')
        df['content'] = df['content'].fillna('')
        
        print(f"✓ Dataset loaded: {len(df)} articles")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Could not find the dataset file: {e}")
        print("Please make sure 'balanced_dataset.csv' exists in the current directory.")
        exit(1)
    except Exception as e:
        print(f"❌ ERROR: An unexpected error occurred while loading dataset: {e}")
        exit(1)

    # Initialize recommender
    print("\n" + "="*70)
    print("INITIALIZING COMPARISON SYSTEM")
    print("="*70)

    try:
        recommender = UrduNewsRecommenderComparison(
            model_name="urduhack/roberta-urdu-small",
            chroma_db_path_full="./chroma_db_mean_content_full_claude",
            chroma_db_path_ae="./chroma_db_mean_content_AE64_claude",
            collection_name_full="urdu_news_embeddings_full_mean_claude",
            collection_name_ae="urdu_news_embeddings_ae64_mean_claude",
            output_doc_path="Urdu_News_Full_vs_Autoencoder_Comparison_Report.docx"
        )
    except Exception as e:
        print(f"❌ ERROR: Failed to initialize recommender: {e}")
        exit(1)

    # Add dataset information to document with NaN handling
    recommender.add_heading("Dataset Information", level=1)
    
    # Get unique categories, handle NaN values
    categories = df['Category'].unique().tolist()
    # Convert all items to string, handle NaN
    categories_str = [str(cat) if pd.notna(cat) else "Unknown" for cat in categories]
    
    dataset_info = [
        ["Total articles", len(df)],
        ["Dataset shape", f"{df.shape}"],
        ["Unique categories", df['Category'].nunique()],
        ["Categories", ", ".join(categories_str[:15]) + ("..." if len(categories_str) > 15 else "")]
    ]
    recommender.add_table(dataset_info, headers=["Metric", "Value"])

    # Display and add system statistics
    print("\n" + "="*70)
    print("SYSTEM STATISTICS")
    print("="*70)
    stats = recommender.get_statistics()
    print(f"Model: {stats['model']}")
    print(f"Device: {stats['device']}")
    print(f"Pooling Method: {stats['pooling_method']}")
    print(f"\nFull Embeddings (768D):")
    for key, value in stats['full_embeddings'].items():
        print(f"  {key}: {value}")
    print(f"\nAutoencoder Embeddings (64D):")
    for key, value in stats['autoencoder_embeddings'].items():
        print(f"  {key}: {value}")

    recommender.add_statistics_to_doc()

    # Define all queries
    queries = [
    "ریاضی اس سوال جواب دے پہلی نظر ریاضی سان سوال لگتا مگر اس انٹرنیٹ متعدد افراد ذہنوں پریشان کرکے رکھ ہےکیا اپنی ریاضی صلاحیت پورا بھروسا ہاں اس سوال جواب جس فیس بک صارفین ذہنوں گھما رکھ جس وجہ اس دو ممکنہ جواب ہیںاسے گو تھمبل تیار کیا اس فیس بک لوگوں ",
    "پاکستان اسٹاک ایکسچینج ملا جلا رجحان ہنڈرڈ انڈیکس پوائنٹس کمی بند پاکستان اسٹاک ایکسچینج اج کاروبار اختتام ملا جلا رجحان دیکھا گیا پاکستان اسٹاک ایکسچینج اج کاروبار اغاز اتار چڑھا جاری ملا جلا رجحان دیکھنے ایا اج کاروبار دوران کمپنیوں کروڑ شیئرز سودے",
    "سام سنگ نئے فلیگ شپ فون تاریخ رونمائی سامنے گئی سام سنگ اپنے نئے فلیگ شپ فون گلیکسی نوٹ ئندہ ماہ متعارف کرانے باضابطہ اعلان کردیا ہےسام سنگ جانب اگست گلیکسی نوٹ متعارف کرایا جائے جس گلیکسی زی فولڈ گلیکسی زی فلپ جی ورژن ایک لائن ایونٹ پیش کیے جانے امک",
    "سلمان خان دوستوں دوستی لے ڈوبی ممبئی ویب ڈیسک بالی وڈ پنڈتوں کہنا بظاہر نظر پروڈکشن میدان اترنے سلمان خان پہلی بار جھٹکا لگنے اپنے دوستوں بچوں عطیہ شیٹی سورج پنچولی ان فلم ہیرو ریلیز ہو چکی اس رپورٹ اچھی نہیں تفصیلات سلمان خان فلم ہیرو اداکارہ کردار ",
    "ئی فون متعارف کرانے تاریخ سامنے گئی رواں سال شروع ایسی اطلاعات سامنے ئی تھیں ئی فون ایٹ اس بار معمول ستمبر سامنے نہیں سکے بلکہ اسے پیش کیا جائے گاتاہم ایپل اس نئے مکمل ری ڈیزائن اسمارٹ فون تاریخ رونمائی سامنے نہیں ئی مگر پتا چل گیا ستمبر عشرے متعارف ",
    "پاکستان ویسٹ انڈیز قسمت بدلنے خواہاں برج ٹان پاکستان عالمی چیمپیئن ویسٹ انڈیز درمیان چار ٹی میچوں سیریز اج اغاز ہونے جا جہاں میزبان ٹیم متحدہ عرب امارات گرین شرٹس بدترین کارکردگی بھلا بہترین کھیل پیش کرنے کوشش کرے گیگزشتہ سال ستمبر ان تین میچوں بے دا",
    "ایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم کراچیایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم ہوگیا کوارٹر فائنل شکست بعد محمد سجاد ابو صائم ایونٹ ہوگئے کراچی جاری ایونٹ کوارٹر فائنل ایران عامر سرکوش مقابلہ پاکستان محمد سجاد ایرانی کیوسٹ عمدہ کارکرد",
    "یو ایس اوپن ٹینس نوواک جوکووچ تیسرے رانڈ پہنچ گئے نیویارک اگست یو ایس اوپن ٹاپ سیڈ دفاعی چیمپئن نوواک جوکووچ باسانی تیسرے رانڈ پہنچ گئے اکہترویں فتح پیٹ سمپراس ہم پلہ بن گئے سرینا ولیمز میڈیسن کیز اپنے میچز سرخرو رہیں نیویارک ٹینس ایرینا جاری سیزن کا",
    "پی ٹی اے زونگ جی اشتہارات واپس لینے ہدایت پاکستان ٹیلی کمیونیکشن اتھارٹی پی ٹی اے چائنا موبائل پاکستان سی پی ایم یعنی زونگ جی ٹیکنالوجی اشتہارات تشہیر روکنے حکم ہےپی ٹی اے جانب جاری بیان زونگ جانب میڈیا حالیہ جی اشتہارات عوام گاہ کیا جاتا کمپنی فائیو",
    "ناقدین نظر بہترین فلمیں ناقدین نظر بہترین فلمیں امریکن گریفیٹی سائیکو سم لائیک اٹ ہاٹ منتھس ویکس اینڈ ڈیز گون ود ونڈ مائی لیفٹ فٹ ہوپ ڈریمز پینز لیبیرنتھ مون لائٹ سنگنگ ان رین تھری کلرز ریڈ بوائے ہڈ کاسا بلانکا گاڈ فادر سیٹزن کین ناقدین نظر بہترین فل",
    "پاکستانی اسکواڈ کرائسٹ چرچ کوئنز ٹان روانہ ہو لاہور دسمبر نیوزی لینڈ موجود قومی کرکٹ اسکواڈ مینیجڈ ئسولیشن چھوڑنے اجازت مل گئی قومی اسکواڈ شامل اکاون ارکان کرائسٹ چرچ مینیجڈ ئسولیشن چھوڑ دو ارکان اپنی ئسولیشن مدت پوری کریں نیوزی لینڈ موجود قومی اسکوا",
    "سرچ ہونے ڈیوائسز پسند ہو نہ ہو مگر ایپل ئی فون سکس ایس گوگل مقبول ڈیوائس قرار دے ہےگوگل جانب ٹرینڈز ٹیکنالوجی فہرست ئی فون سکس ایس سرچ ہونے باعث سرفہرست رہاویسے حیرت انگیز نہیں ئی فون نئے ماڈلز لگ بھگ سال بہترین قرار پاتے لوگوں نئے ئی فون جاننے تجسس ",
    "پاکستان اسٹاک ایکسچینج پوائنٹس اضافہ کراچی اپریل کاروباری ہفتے تیسرے روز پاکستان اسٹاک ایکسچینج مثبت رجحان انڈیکس پوائنٹس بہتری بدھ پاکستان اسٹاک ایکسچینج بہتری رجحان ایس ای انڈیکس پوائنٹس اضافے بعد ہزار پوائنٹس سطح پہنچ گیا کاروبار دوران کروڑ لاکھ ہ",
     ]

    # Execute all queries
    for idx, query in enumerate(queries, 1):
        print("\n\n" + "="*70)
        print(f"EXECUTING QUERY {idx} OF {len(queries)}")
        print("="*70)

        try:
            results = recommender.get_recommendations_comparison(
                query=query,
                n_results=50
            )

            recommender.compare_and_visualize(results, df=df, query_number=idx)
        except Exception as e:
            print(f"❌ ERROR: Failed to execute query {idx}: {e}")
            recommender.add_heading(f"Query {idx}: Failed", level=1)
            recommender.add_paragraph(f"Error: {str(e)}")
            continue

    # Save the final document
    try:
        recommender.save_document()
    except Exception as e:
        print(f"❌ ERROR: Failed to save document: {e}")

    print("\n" + "="*70)
    print("✓ COMPARISON SYSTEM DEMO COMPLETED!")
    print("✓ Full embeddings: 768 dimensions")
    print("✓ Autoencoder embeddings: 64 dimensions (Neural Network)")
    print(f"✓ All {len(queries)} queries executed")
    print(f"✓ Comprehensive report saved to: Urdu_News_Full_vs_Autoencoder_Comparison_Report.docx")
    print("✓ Visualizations saved as PNG files")
    print("="*70)

URDU NEWS RECOMMENDATION COMPARISON SYSTEM
Full Embeddings (768D) vs Autoencoder Embeddings (64D)
OUTPUTTING TO WORD DOCUMENT

Loading dataset for content display...
Cleaning dataset...
✓ Dataset loaded: 111853 articles

INITIALIZING COMPARISON SYSTEM
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Loading Autoencoder model from: chroma_db_mean_content_AE64_claude/autoencoder_model.pkl
✓ Autoencoder model loaded (64 dimensions)
  Final training loss: 0.027751269611770827
Connecting to Full Embeddings ChromaDB at: chroma_db_mean_content_full_claude
✓ Connected to full embeddings collection: urdu_news_embeddings_full_mean_claude
✓ Total articles in full database: 111853
Connecting to Autoencoder Embeddings ChromaDB at: chroma_db_mean_content_AE64_claude
✓ Connected to Autoencoder embeddings collection: urdu_news_embeddings_ae64_mean_claude
✓ Total articles in Autoencoder database: 111853

SYSTEM STATISTICS
Model: urduhack/roberta-urdu-small
Device: cuda
Pooling Method: MEAN